### This notebook uses features from our previous notebook.
- **GP + ARD** : https://www.kaggle.com/code/illuminatep/kasa-house-pricing-with-gp-ard

# 0. Introduction

### * Built with PriorLabs-TabPFN & TabICL

Figure 1 : **(a)** The pipeline. **(b)** One fold of the ensemble. Two inputs, each made of the 121 base features plus gene columns from saved GP models, feed three experts, and the sale condition of each house decides whose prediction it gets.

그림 1 : **(a)** 전체 파이프라인. **(b)** 앙상블의 fold 하나. 기본 피쳐 121열에 저장된 GP 모델의 유전자 열을 붙인 입력 두 가지가 전문가 셋에게 들어가고, 집마다 판매 조건에 따라 어느 전문가의 예측을 쓸지가 정해진다.

### **Summary**
This notebook takes the columns chosen by multi-gene GPs evolved with a ridge decider, feeds them to two tabular foundation models, TabPFN v2 and TabICL v2, and ensembles the two differently on each sale-condition axis.

1. **Columns chosen by the GP** : the GPs were evolved earlier, with a ridge or ARD (a ridge with one penalty per gene) as the decider. Each gene they kept is a small expression over the design columns, and its values become an extra column. Nothing is evolved in this notebook.
2. **Two foundation models** : the gene columns are added to 121 base features and given to TabPFN v2 and TabICL v2. Neither model is trained on this data. Each takes the fold's train rows as context and predicts every house in a single pass (in-context learning).
3. **A different ensemble on each axis** : the sale condition splits the houses into market and non-market sales. Market houses get the mean of TabPFN and TabICL, and non-market houses are left to TabICL alone, so each fold has three experts.

<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">Sale condition</th><th style="text-align:center">Houses (test / valid)</th><th style="text-align:left">Predicted by</th><th style="text-align:left">Context</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>Market</b> (Normal, Partial)</td><td style="text-align:center">1,324 / 1,314</td><td style="text-align:left">0.5 · TabPFN v2 + 0.5 · TabICL v2</td><td style="text-align:left">the fold's 1,044–1,061 market train rows</td></tr>
<tr><td style="text-align:left"><b>Non-market</b> (Abnorml, Family, Alloca, AdjLand)</td><td style="text-align:center">135 / 137</td><td style="text-align:left">TabICL v2 alone</td><td style="text-align:left">the fold's 101–119 non-market train rows</td></tr>
</tbody></table>
</div><br>

Each fold's predictions are clipped to that fold's training log-price range, and the submission is the mean of the five folds in log space.<br>
The recipe was picked by a grid search over three choices: which genes each model gets, which train rows it takes as context, and how the two predictions are mixed.<br>
This notebook runs the best combination and nothing else.

This notebook uses TabPFN v2 (Hollmann et al., 2025; Prior Labs License 1.1) and TabICL v2 (Qu et al., 2026; BSD-3-Clause).<br>
Both pretrained weights are downloaded at run time from the authors' official repositories and checked against a fixed SHA-256, and they are not redistributed with this notebook.

### **What the experts get**
- **121 base features** : the 78 numeric columns of the GP design (section 3) and the raw labels of 43 categorical columns.<br>
  An ordinal column appears twice, once as its ordered number and once as its raw label.
- **Gene columns** : a gene is a small expression that a GP evolved over the 89 design columns, such as `min(log(LotArea), add(BsmtFinSF1, SaleCondition_TE))`.<br>
  The bundle holds four GP models per fold, and their genes are evaluated on every row and added as columns.<br>
  Only the gene values go in, not the GP's coefficients, so combining them is left to the experts.
  - TabPFN gets the **per-axis genes** of two GP + ridge models, one evolved on market sales and one on non-market sales (41–52 columns per fold).
  - TabICL gets the **ARD · ridge genes** of a GP + ARD and a GP + ridge, both evolved on all sales (43–51 columns per fold).
- On rows a GP model never trained on, each gene is clipped to the range it had on its training rows, the same guard the GP itself used.

### **Two foundation models**
Both are transformers pretrained on millions of synthetic tables.<br>
Given train rows with their prices, they return a price distribution for each row to predict, and no gradient step ever touches this data.

<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left"></th><th style="text-align:left">TabPFN v2</th><th style="text-align:left">TabICL v2</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>Reads the table</b></td><td style="text-align:left">cell by cell, with 12 blocks that alternate attention within a row and within a column</td><td style="text-align:left">row by row: each row is first compressed into one vector, then 12 blocks of in-context learning run over the row vectors</td></tr>
<tr><td style="text-align:left"><b>Predicts</b></td><td style="text-align:left">probabilities over 5,000 price bins</td><td style="text-align:left">999 price quantiles</td></tr>
<tr><td style="text-align:left"><b>Size</b></td><td style="text-align:left">11 M parameters</td><td style="text-align:left">28.5 M parameters</td></tr>
<tr><td style="text-align:left"><b>Members</b></td><td style="text-align:left">the mean of 8 readings of the table (feature and target transforms × column orders)</td><td style="text-align:left">the mean of 8 readings (2 normalisations × 4 column orders)</td></tr>
</tbody></table>
</div><br>

The two are about equally accurate on market sales but miss on different houses, and the mean of two such models beats either one (Krogh &amp; Vedelsby 1995).<br>
So a market house gets the plain mean of the two, while a non-market house, with a context of only 101–119 rows, is left to TabICL.

### **Why split by sale condition**
The data documentation lists six sale conditions (De Cock 2011).<br>
In four of them the price is set outside the open market: an abnormal sale (trade, foreclosure, short sale), a sale between family members, the allocation of two linked properties, and the purchase of adjoining land.<br>
These make up 9% of the training rows, and their prices follow rules of their own.

So each expert takes the train rows of one group only.<br>
A foreclosure cannot pull the market experts around, and the non-market expert never averages a family sale with ordinary ones.

### **Guards**
- **Weights** : downloaded on every run and checked against a fixed SHA-256, so a changed file stops the notebook. They stay in a temporary folder and never reach the output.
- **Bundle** : the 20 GP model files are our own upload, so they are only checked for existence. Each GP model must still reproduce its recorded validation score from the gene values this notebook computes (section 6).
- **Unseen rows** : a gene's value is clipped to its training range, and every prediction to the fold's training log-price range.
- **Id 524 and 1299** : houses of 4,676 and 5,642 sq ft sold as partial sales for 184,750 and 160,000 dollars. As in the GP + ARD notebook, they stay on the train side of every fold and out of the price statistics.

### **What changed since the GP + ARD notebook**
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left"></th><th style="text-align:left">GP + ARD notebook</th><th style="text-align:left">This notebook</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>Model</b></td><td style="text-align:left">multi-gene GP + ARD</td><td style="text-align:left">TabPFN v2 and TabICL v2, mixed by sale condition</td></tr>
<tr><td style="text-align:left"><b>GP</b></td><td style="text-align:left">evolved in the notebook, about 10 hours on four cores</td><td style="text-align:left">four saved GP models per fold, read from the bundle</td></tr>
<tr><td style="text-align:left"><b>What the GP gives</b></td><td style="text-align:left">the prediction (genes × ARD weights)</td><td style="text-align:left">gene values only, as extra columns</td></tr>
<tr><td style="text-align:left"><b>Features</b></td><td style="text-align:left">89 design columns</td><td style="text-align:left">121 base features + 41–52 or 43–51 gene columns</td></tr>
<tr><td style="text-align:left"><b>Training rows</b></td><td style="text-align:left">all 1,453, five folds</td><td style="text-align:left">the same folds, with each expert taking one sale group's train rows as context</td></tr>
<tr><td style="text-align:left"><b>Accelerator</b></td><td style="text-align:left">none (CPU)</td><td style="text-align:left">GPU</td></tr>
<tr><td style="text-align:left"><b>Run time</b></td><td style="text-align:left">about 10 hours</td><td style="text-align:left">minutes</td></tr>
<tr><td style="text-align:left"><b>Submission</b></td><td style="text-align:left">mean of five fold models in log space</td><td style="text-align:left">mean of five fold predictions in log space</td></tr>
</tbody></table>
</div><br>

<hr>

### **GP가 고른 열을 TabPFN · TabICL에 넣고, 판매 조건 축마다 다르게 앙상블한다**
이 노트북은 GP + 릿지로 진화시킨 다유전자 GP가 고른 열을 표 파운데이션 모델 TabPFN v2와 TabICL v2의 입력으로 쓰고, 두 모델을 판매 조건 축마다 다르게 앙상블한다.<br>
순서대로 보면 이렇다.

1. **GP가 고른 열** : 다유전자 GP는 앞서 진화시켜 둔 것이고, 결정자는 릿지 또는 ARD(유전자마다 벌점을 따로 두는 릿지)다. GP가 남긴 유전자는 설계행렬 열로 만든 작은 식이고, 이 식의 값을 열로 붙인다.<br>
이 노트북에서 진화를 다시 돌리지 않는다.
2. **두 파운데이션 모델** : 유전자 열을 기본 피쳐 121열에 붙여 TabPFN v2와 TabICL v2에 넣는다. 두 모델은 이 데이터로 가중치를 학습하지 않는다. fold의 train 행을 컨텍스트로 넣으면 예측할 집 전부의 가격을 한 번에 낸다(in-context learning).
3. **축마다 다른 앙상블** : 판매 조건으로 집을 시장 거래와 비시장 거래로 나눈다. 시장 거래 집은 TabPFN과 TabICL의 예측을 반반 섞고, 비시장 거래 집은 TabICL 혼자 맡는다. 그래서 fold마다 전문가가 셋이다.

<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">판매 조건</th><th style="text-align:center">집 수(test / valid)</th><th style="text-align:left">예측하는 모델</th><th style="text-align:left">컨텍스트</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>시장 거래</b>(Normal, Partial)</td><td style="text-align:center">1,324 / 1,314</td><td style="text-align:left">0.5 · TabPFN v2 + 0.5 · TabICL v2</td><td style="text-align:left">그 fold의 시장 거래 train 행(1,044~1,061행)</td></tr>
<tr><td style="text-align:left"><b>비시장 거래</b>(Abnorml, Family, Alloca, AdjLand)</td><td style="text-align:center">135 / 137</td><td style="text-align:left">TabICL v2 단독</td><td style="text-align:left">그 fold의 비시장 거래 train 행(101~119행)</td></tr>
</tbody></table>
</div><br>

fold마다 예측값을 그 fold의 train log 가격 범위로 자르고, 다섯 fold의 예측을 log 공간에서 평균해 제출한다.<br>
어느 모델에 어떤 유전자를 주고, 어느 train 행을 컨텍스트로 쓰고, 두 예측을 어떻게 섞을지는 그리드 서치로 가장 나은 조합을 골랐다.<br>
이 노트북은 그 조합만 실행한다.

TabPFN v2(Hollmann 외 2025, Prior Labs License 1.1)와 TabICL v2(Qu 외 2026, BSD-3-Clause)를 쓴다.<br>
두 모델의 가중치는 실행할 때마다 공식 저장소에서 받아 고정해 둔 SHA-256과 대조한다.<br>

### **전문가에게 들어가는 입력**
- **기본 피쳐 121열** : 3절 GP 설계행렬의 수치 78열과, 범주형 열 43개의 원래 값이다.<br>
  순서형 열은 순서를 매긴 숫자와 원래 값으로 두 번 들어가므로, 모델은 순서와 범주를 둘 다 본다.
- **유전자 열** : 유전자는 GP가 설계행렬 89열로 진화시킨 작은 식이다(예: `min(log(LotArea), add(BsmtFinSF1, SaleCondition_TE))`).<br>
  번들에는 fold마다 GP 모델이 네 개씩 있고, 이 모델들의 유전자를 모든 행에서 계산해 열로 붙인다.<br>
  GP의 계수는 쓰지 않고 유전자 값만 넣는다.<br>
  값을 어떻게 조합할지는 전문가가 정한다.
  - TabPFN에는 **축별 유전자**(fold당 41\~52열)를 준다. 시장 거래로만 진화한 GP + 릿지와 비시장 거래로만 진화한 GP + 릿지의 유전자를, 시장 · 비시장 행 모두에 같은 열로 넣는다.
  - TabICL에는 **ARD · 릿지 유전자**(fold당 43\~51열)를 준다. 이전 노트북과 같은 셋업으로 전체 거래에서 진화한 GP + ARD와, 결정자만 릿지로 바꾼 같은 셋업의 GP가 만든 유전자다.
- GP 모델이 학습하지 않은 행에서는 유전자 값을 그 유전자가 train 행에서 낸 범위로 자른다. GP가 쓰던 안전장치를 그대로 쓴 것이다.

### **두 파운데이션 모델**
둘 다 합성 데이터로 만든 수많은 표로 미리 학습한 트랜스포머다.<br>
가격이 붙은 train 행을 표째로 넣으면 예측할 행마다 가격 분포를 돌려준다.<br>
이 데이터로는 경사 하강을 한 번도 하지 않는다.

<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left"></th><th style="text-align:left">TabPFN v2</th><th style="text-align:left">TabICL v2</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>표를 읽는 방식</b></td><td style="text-align:left">칸 단위로 읽는다. 12블록 동안 같은 집의 칸끼리, 같은 열의 다른 집끼리 번갈아 어텐션을 계산한다</td><td style="text-align:left">행 단위로 읽는다. 행마다 먼저 벡터 하나로 줄인 뒤, 행 벡터끼리 12블록 동안 in-context learning을 한다</td></tr>
<tr><td style="text-align:left"><b>예측 형태</b></td><td style="text-align:left">가격 구간 5,000개의 확률</td><td style="text-align:left">가격 분위수 999개</td></tr>
<tr><td style="text-align:left"><b>크기</b></td><td style="text-align:left">매개변수 1,108만 개</td><td style="text-align:left">매개변수 2,850만 개</td></tr>
<tr><td style="text-align:left"><b>멤버</b></td><td style="text-align:left">표를 조금씩 바꿔 읽은 예측 8개의 평균(피쳐 · 타깃 변환 × 열 순서)</td><td style="text-align:left">예측 8개의 평균(정규화 2가지 × 열 순서 4가지)</td></tr>
</tbody></table>
</div><br>

두 모델은 시장 거래에서 정확도가 비슷한데, 틀리는 집이 서로 다르다.<br>
이런 두 모델을 평균하면 각자보다 낫다.<br>
앙상블의 오차는 구성원 오차의 평균에서 구성원끼리 갈리는 정도를 뺀 값이기 때문이다(Krogh · Vedelsby 1995).<br>
그래서 시장 거래 집에는 두 모델의 단순 평균을 쓴다.<br>
비시장 거래 집은 컨텍스트가 101\~119행뿐이라 TabICL에게 맡긴다.

### **판매 조건으로 나누는 이유**
데이터 설명서에는 판매 조건이 여섯 가지 있다(De Cock 2011).<br>
그중 넷은 비정상 거래(교환 · 압류 · 급매), 가족 간 거래, 묶인 두 부동산의 할당, 옆 땅 매입으로, 가격이 열린 시장 밖에서 정해진다.<br>
이런 거래는 train 행의 9%이고, 가격이 정해지는 방식이 보통 거래와 다르다.

그래서 전문가마다 한 묶음의 train 행만 컨텍스트로 쓴다.<br>
시장 거래 전문가는 압류 거래에 휘둘리지 않고, 비시장 거래 전문가는 가족 간 거래를 보통 거래와 한데 섞어 평균하지 않는다.

### **안전장치**
- **가중치** : 실행할 때마다 받아서 고정해 둔 SHA-256과 대조하고, 파일이 바뀌었으면 노트북이 멈춘다. 받은 파일은 임시 폴더에만 두고 출력에는 넣지 않는다.
- **번들** : GP 모델 파일 20개는 직접 올린 것이라 있는지만 확인한다. 대신 GP 모델마다, 이 노트북이 계산한 유전자 값으로 기록된 valid 점수를 재현해야 한다(6절).
- **처음 보는 행** : 유전자 값은 그 유전자의 train 범위로, 예측값은 그 fold의 train log 가격 범위로 자른다.
- **이상치 두 채** : Id 524와 1299(4,676 · 5,642sqft, 신축 분양 거래로 184,750 · 160,000달러)는 이전 노트북처럼 모든 fold에서 train 쪽에 두고, 가격 통계에서는 뺀다.

### **이전 노트북과 달라진 점**
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left"></th><th style="text-align:left">이전 노트북(GP + ARD)</th><th style="text-align:left">이 노트북</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>모델</b></td><td style="text-align:left">다유전자 GP + ARD</td><td style="text-align:left">TabPFN v2와 TabICL v2를 판매 조건별로 섞음</td></tr>
<tr><td style="text-align:left"><b>GP</b></td><td style="text-align:left">노트북 안에서 진화(4코어에서 약 10시간)</td><td style="text-align:left">fold마다 저장된 GP 모델 네 개를 번들에서 읽음</td></tr>
<tr><td style="text-align:left"><b>GP가 내는 것</b></td><td style="text-align:left">예측값(유전자 × ARD 계수)</td><td style="text-align:left">유전자 값만 추가 열로</td></tr>
<tr><td style="text-align:left"><b>피쳐</b></td><td style="text-align:left">설계행렬 89열</td><td style="text-align:left">기본 피쳐 121열 + 유전자 41~52열 또는 43~51열</td></tr>
<tr><td style="text-align:left"><b>train 행</b></td><td style="text-align:left">1,453행 전부, 5 fold</td><td style="text-align:left">같은 fold 표. 전문가마다 한 판매 조건 묶음의 train 행을 컨텍스트로 씀</td></tr>
<tr><td style="text-align:left"><b>가속기</b></td><td style="text-align:left">없음(CPU)</td><td style="text-align:left">GPU</td></tr>
<tr><td style="text-align:left"><b>실행 시간</b></td><td style="text-align:left">약 10시간</td><td style="text-align:left">몇 분</td></tr>
<tr><td style="text-align:left"><b>제출</b></td><td style="text-align:left">fold 모델 5개의 log 공간 평균</td><td style="text-align:left">fold 예측 5개의 log 공간 평균</td></tr>
</tbody></table>
</div><br>

# 1. Import

### Install the two foundation-model packages / 파운데이션 모델 패키지 설치

두 파운데이션 모델 패키지(`tabpfn` 9.2.0 · `tabicl` 2.2.0)를 버전을 고정해 설치한다.<br>
같은 버전이 이미 있으면 건너뛴다.<br>
설치 과정에서는 numpy를 불러오지 않으므로, 다음 셀의 BLAS 스레드 설정보다 먼저 와도 된다.

In [1]:
# The two foundation-model packages, pinned. tabpfn 9.2.0 defaults to a newer model, so section 4 names v2 explicitly.
# 두 파운데이션 모델 패키지를 버전을 고정해 설치한다. tabpfn 9.2.0은 기본 모델이 v2가 아니라서 4절에서 v2를 직접 지정한다.
import importlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

PINS = {'tabpfn': '9.2.0', 'tabicl': '2.2.0'}


def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


_need = [f'{k}=={v}' for k, v in PINS.items() if installed(k) != v]
if _need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_need], check=True)
    importlib.invalidate_caches()

for _k, _v in PINS.items():
    assert installed(_k) == _v, f'{_k} {installed(_k)} is installed, not {_v}!!'
    print(f'{_k:<7}: {_v}')

tabpfn : 9.2.0
tabicl : 2.2.0


### Import

In [2]:
# Thread caps must be set BEFORE numpy is imported, or BLAS will already have spawned its pool.
# BLAS 스레드 상한은 **numpy import 전에** 걸어야 한다. 늦으면 이미 스레드 풀이 만들어져 있다.
import os

for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
           'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[_v] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'      # 내려받기 진행 막대 off

import hashlib, json, operator, random, shutil, tempfile, time, zipfile
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

# Turn off all warnings
# 경고 off
import warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import numpy as np
import pandas as pd
import torch
from deap import gp
from huggingface_hub import hf_hub_download
from sklearn.model_selection import KFold
from tabicl import TabICLRegressor
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

# Belt and braces: if anything loaded numpy before this cell, the variables above came too late. threadpoolctl caps
# the BLAS that is actually loaded. One thread keeps every sum in the same order, TabPFN's preprocessing included.
# 이 셀보다 먼저 numpy가 import됐다면 위 환경 변수는 이미 늦다. 그래서 threadpoolctl로 실제 불러온 BLAS를 1스레드로 묶는다.
# 스레드가 하나여야 덧셈 순서가 늘 같고, TabPFN의 전처리도 매번 같은 값을 낸다.
from threadpoolctl import threadpool_info, threadpool_limits
threadpool_limits(1)

print('numpy  :', np.__version__)
print('pandas :', pd.__version__)
print('torch  :', torch.__version__)
print('deap   : ok')

numpy  : 2.1.3
pandas : 2.3.3
torch  : 2.11.0+cu128
deap   : ok


### Environment check / 환경 점검

In [3]:
# The two foundation models run on the GPU. TabPFN refuses a CPU context above 1,000 rows, and the market context is larger.
# 두 파운데이션 모델은 GPU에서 돈다. TabPFN은 CPU에서는 컨텍스트가 1,000행을 넘으면 돌지 않는데, 시장 거래 컨텍스트가 그보다 크다.
N_CPU = os.cpu_count() or 1

print('cpu count       :', N_CPU)
print('thread cap      :', os.environ['OMP_NUM_THREADS'], '(per process)')
print('BLAS in effect  :', ', '.join(f"{i['internal_api']} {i['num_threads']} thread(s)" for i in threadpool_info()))
try:
    _mem = int(next(l.split()[1] for l in open('/proc/meminfo') if l.startswith('MemAvailable'))) / 1e6
    print(f'memory available: {_mem:.1f} GB')
except Exception:
    pass

assert torch.cuda.is_available(), 'no GPU found, turn the accelerator on in the notebook settings!!'
_gpu = torch.cuda.get_device_properties(0)
print(f'GPU             : {_gpu.name} · {_gpu.total_memory / 1e9:.1f} GB')

cpu count       : 4
thread cap      : 1 (per process)
BLAS in effect  : openblas 1 thread(s), openmp 1 thread(s), openblas 1 thread(s), openmp 1 thread(s)
memory available: 31.2 GB
GPU             : Tesla T4 · 15.6 GB


### Directory setup / 경로 설정

In [4]:
# 경로 설정
if Path('/kaggle/input').exists():
    # when Kaggle
    ROOT_DIR = Path('/kaggle/working')
    COMP_DIR = next(p.parent for p in sorted(Path('/kaggle/input').rglob('train.csv'))
                    if (p.parent / 'test.csv').exists())      # 대회 데이터 / competition data
    BUNDLE_DIR = next(p.parent for p in sorted(Path('/kaggle/input').rglob('folds.csv')))     # fold 표 + GP 모델 / the bundle
    print('Environment : Kaggle')
else:
    # when local: point HP_COMP_DIR and HP_BUNDLE_DIR at the inputs, or keep them next to the notebook
    # 로컬에서는 입력 위치를 HP_COMP_DIR · HP_BUNDLE_DIR로 주거나, 입력을 노트북 옆에 둔다
    ROOT_DIR = Path(os.environ.get('HP_KAGGLE_WORK', '.'))
    COMP_DIR = Path(os.environ.get('HP_COMP_DIR', '.'))
    BUNDLE_DIR = Path(os.environ.get('HP_BUNDLE_DIR', '.'))
    print('Environment : local')

# The pretrained weights go to a temporary folder outside the output. The notebook downloads them and does not ship them.
# 사전 학습 가중치는 출력 폴더 밖의 임시 폴더에 받는다. 노트북은 가중치를 받아서 쓸 뿐 함께 내보내지 않는다.
WEIGHT_DIR = Path(tempfile.gettempdir()) / 'weights'
OUTPUT_DIR = ROOT_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUB_PATH = ROOT_DIR / 'submission.csv'

print(f'ROOT_DIR   = {ROOT_DIR}')
print(f'COMP_DIR   = {COMP_DIR}')
print(f'BUNDLE_DIR = {BUNDLE_DIR}')
print(f'WEIGHT_DIR = {WEIGHT_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

Environment : Kaggle
ROOT_DIR   = /kaggle/working
COMP_DIR   = /kaggle/input/competitions/house-prices-advanced-regression-techniques
BUNDLE_DIR = /kaggle/input/datasets/illuminatep/hp-bundle
WEIGHT_DIR = /tmp/weights
OUTPUT_DIR = /kaggle/working/outputs


### Check assets / 자산 검사

In [5]:
# 자산 검사
# The inputs only have to exist: the competition files, the fold table and the 20 GP models of the bundle.
# The only files checked by hash are the two pretrained weights in the next cell, which we did not make.
# 입력 파일은 있는지만 확인한다. 대회 파일, fold 표, 번들의 GP 모델 20개다.
# 해시로 확인하는 것은 다음 셀에서 받는 사전 학습 가중치 두 개뿐이다. 번들은 직접 올린 것이라 따로 검사하지 않는다.
FOLDS_PATH = BUNDLE_DIR / 'folds.csv'
GP_RUNS = ('ard', 'ridge', 'market', 'nonmarket')
GP_FILES = {(r, f): BUNDLE_DIR / f'gp_{r}_fold{f}.json' for r in GP_RUNS for f in range(5)}

_bad = []
for _f in ('train.csv', 'test.csv', 'sample_submission.csv'):
    if not (COMP_DIR / _f).exists():
        _bad.append(f'Missing {_f} ({COMP_DIR}), attach the competition data')
if not FOLDS_PATH.exists():
    _bad.append(f'Missing folds.csv ({BUNDLE_DIR}), attach the bundle dataset')
_missing = [p.name for p in GP_FILES.values() if not p.exists()]
if _missing:
    _bad.append(f'Missing {len(_missing)} of {len(GP_FILES)} GP model files in {BUNDLE_DIR} (e.g. {_missing[0]}), '
                f'attach the bundle version with the GP models')

assert not _bad, 'Asset check failed!!:\n  ' + '\n  '.join(_bad)
print('*** Asset check passed ***')
print(f'    competition files · folds.csv · {len(GP_FILES)} GP model files')

*** Asset check passed ***
    competition files · folds.csv · 20 GP model files


### Pretrained weights / 사전 학습 가중치

- **TabPFN v2** : Hugging Face `Prior-Labs/TabPFN-v2-reg`의 `tabpfn-v2-regressor.ckpt`(44MB)
- **TabICL v2** : Hugging Face `jingang/TabICL`의 `tabicl-regressor-v2-20260212.ckpt`(114MB)

두 파일은 밖에서 받아 오는 것이라, 실행할 때마다 공식 저장소에서 새로 받고 아래 셀에 적어 둔 SHA-256과 대조한다.<br>
해시를 확인하는 파일은 이 둘뿐이다.<br>
번들은 직접 올린 것이라 바로 위 셀에서 파일이 있는지만 확인했다.<br>
저장소의 파일이 바뀌면 여기서 멈춘다.<br>
다른 가중치로 계속 돌리면 같은 노트북이라도 다른 예측이 나오기 때문이다.<br>
받은 파일은 출력 폴더 밖의 임시 폴더에 둔다.

In [6]:
# The two pretrained weights are not ours, so they are downloaded on every run from the authors' official Hugging Face
# repositories and checked against a fixed SHA-256. A changed file stops the notebook here.
# 두 사전 학습 가중치는 밖에서 받아 오는 파일이라, 실행할 때마다 저자들의 공식 Hugging Face 저장소에서 받아 고정해 둔 SHA-256과 대조한다.
# 파일이 바뀌었으면 여기서 멈춘다. 받은 파일은 WEIGHT_DIR에만 두고 출력 폴더에는 넣지 않는다.
WEIGHTS = {
    'tabpfn': dict(repo_id='Prior-Labs/TabPFN-v2-reg', filename='tabpfn-v2-regressor.ckpt',
                   sha256='2ab5a07d5c41dfe6db9aa7ae106fc6de898326c2765be66505a07e2868c10736'),
    'tabicl': dict(repo_id='jingang/TabICL', filename='tabicl-regressor-v2-20260212.ckpt',
                   sha256='0db9cb538f114e79026bf08f45f41ad8dd7ad2de2aaca9a5ca8cd3bd9748ae7a'),
}


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


WEIGHT_PATH = {}
for _k, _w in WEIGHTS.items():
    _t = time.time()
    _p = Path(hf_hub_download(repo_id=_w['repo_id'], filename=_w['filename'],
                              local_dir=WEIGHT_DIR / _k, force_download=True))
    _h = file_sha256(_p)
    assert _h == _w['sha256'], f'{_w["filename"]} has SHA-256 {_h}, not the weights this notebook was written for!!'
    assert ROOT_DIR.resolve() not in _p.resolve().parents, f'{_p} is inside the output folder!!'
    WEIGHT_PATH[_k] = _p
    print(f'{_k:<7}: {_w["repo_id"]} / {_w["filename"]} · {_p.stat().st_size / 1e6:.1f} MB '
          f'· SHA-256 ok · {time.time() - _t:.1f} s')

tabpfn : Prior-Labs/TabPFN-v2-reg / tabpfn-v2-regressor.ckpt · 44.4 MB · SHA-256 ok · 1.4 s
tabicl : jingang/TabICL / tabicl-regressor-v2-20260212.ckpt · 114.3 MB · SHA-256 ok · 1.3 s


### Set seed and recipe / 시드와 레시피

In [7]:
# It must be 42, since it is the ultimate answer to life, the universe and everything
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f'Seed = {SEED}')

# The material (section 3): one column out, two houses pinned to the train side and kept out of the price statistics.
# 재료(3절): 열 하나를 빼고, 집 두 채를 train 쪽에 고정하고 가격 통계에서 뺀다.
DROP_COLS = ('Utilities',)
PIN_IDS = (524, 1299)
STAT_EXCLUDE_IDS = (524, 1299)

# The sale-condition axis (section 3): market sales, and the four conditions that set a price outside the open market.
# 판매 조건 축(3절): 시장 거래와, 가격이 열린 시장 밖에서 정해지는 네 가지 판매 조건.
MARKET = ('Normal', 'Partial')
NONMARKET = ('Abnorml', 'Family', 'Alloca', 'AdjLand')

# The recipe (section 4): which genes and which train rows (the context) each model gets, and how the predictions are mixed.
# 레시피(4절): 모델마다 어떤 유전자와 어느 train 행(컨텍스트)을 주는지, 그리고 예측을 어떻게 섞는지.
GENE_SETS = {'axis': ('market', 'nonmarket'),      # 축별 유전자: 판매 조건별로 진화한 GP + 릿지 두 개
             'ard_ridge': ('ard', 'ridge')}        # ARD · 릿지 유전자: 전체 거래로 진화한 GP + ARD와 GP + 릿지
EXPERTS = {'tabpfn_market': dict(model='tabpfn', genes='axis', context='market'),
           'tabicl_market': dict(model='tabicl', genes='ard_ridge', context='market'),
           'tabicl_nonmarket': dict(model='tabicl', genes='ard_ridge', context='nonmarket')}
W_MARKET = 0.5           # 시장 거래 행에서 TabPFN의 몫. 나머지는 TabICL이고, 비시장 거래 행은 TabICL만 쓴다
N_MEMBERS = 8            # TabICL이 평균하는 멤버 수. TabPFN은 'auto'로 두며 v2에서는 8이다
MODEL_SEED = 0           # 두 모델의 random_state

for _k, _s in EXPERTS.items():
    print(f'{_k:<16}: {_s["model"]} · genes {"+".join(GENE_SETS[_s["genes"]])} · context {_s["context"]} train rows')
print(f'mix: market rows {W_MARKET} · TabPFN + {1 - W_MARKET} · TabICL, non-market rows TabICL alone')

Seed = 42
tabpfn_market   : tabpfn · genes market+nonmarket · context market train rows
tabicl_market   : tabicl · genes ard+ridge · context market train rows
tabicl_nonmarket: tabicl · genes ard+ridge · context nonmarket train rows
mix: market rows 0.5 · TabPFN + 0.5 · TabICL, non-market rows TabICL alone


# 2. Load the bundle / 데이터셋 불러오기

입력 데이터셋(번들)에는 두 가지가 들어 있다.<br>
행 데이터는 대회 `train.csv`에서 Id로 찾아온다.

- **fold 표** : `Id, fold` 두 열, 1,453행이다. 이전 노트북과 같은 표다.
- **GP 모델 20개** : GP 런 4개 × fold 5개다. fold f의 전문가에게는 fold f의 GP 모델 유전자만 넣는다.

fold에는 쓸 수 있는 train 행을 모두 나눠 넣는다.<br>
모순된 7행을 지운 1,453행이고, 따로 떼어 두는 holdout은 없다.<br>
점수는 리더보드에서 읽는다.

- **valid 행** : fold마다 290\~291행이다. 전문가의 컨텍스트에는 넣지 않고, 6절의 점검과 8절의 점수 문턱에만 쓴다.
- **이상치 두 채** : Id 524 · 1299는 어느 fold에서도 valid 행이 되지 않는다(3절).

In [8]:
# The bundle holds the fold table and the 20 saved GP models. The rows themselves come from the competition csv.
# 번들에는 fold 표와 저장된 GP 모델 20개가 있다. 행 데이터는 대회 csv에서 가져온다.
NA_OPTS_LOAD = dict(keep_default_na=False, na_values=['NA'])    # 'NA'만 결측으로 / only 'NA' is missing

train_csv = pd.read_csv(COMP_DIR / 'train.csv', **NA_OPTS_LOAD)
test_csv = pd.read_csv(COMP_DIR / 'test.csv', **NA_OPTS_LOAD)
fold_tbl = pd.read_csv(FOLDS_PATH)

assert list(fold_tbl.columns) == ['Id', 'fold'], f'fold table columns {list(fold_tbl.columns)}!!'
assert fold_tbl.Id.is_unique, 'duplicated Id in the fold table!!'
assert fold_tbl.Id.isin(train_csv.Id).all(), \
    f'{int((~fold_tbl.Id.isin(train_csv.Id)).sum())} fold ids are not in the competition train.csv!!'

FOLD_OF = dict(zip(fold_tbl.Id, fold_tbl.fold))
print(f'train.csv {len(train_csv):,} rows · test.csv {len(test_csv):,} rows')
print(f'fold table {len(fold_tbl):,} rows · sizes {np.bincount(fold_tbl.fold).tolist()}')

# One GP model per run and fold: its genes, its decider's solution, and the sales it was evolved on.
# 런마다, fold마다 GP 모델이 하나씩 있다. 유전자와 결정자의 해, 진화에 쓴 판매 조건이 들어 있다.
GP_REC = {k: json.loads(p.read_text(encoding='utf-8')) for k, p in GP_FILES.items()}
for (_r, _f), _rec in GP_REC.items():
    assert (_rec['run'], _rec['fold']) == (_r, _f), f'{GP_FILES[_r, _f].name} holds run {_rec["run"]} fold {_rec["fold"]}!!'

print(f'\n{"GP model":<11}{"decider":<9}{"evolved on":<12}live genes, fold 0~4')
for _r in GP_RUNS:
    _rec = GP_REC[_r, 0]
    print(f'{_r:<11}{_rec["decider"]:<9}{_rec["rows"] + " sales":<12}'
          + ', '.join(str(int(np.sum(GP_REC[_r, _f]['alive']))) for _f in range(5)))

train.csv 1,460 rows · test.csv 1,459 rows
fold table 1,453 rows · sizes [291, 291, 291, 290, 290]

GP model   decider  evolved on  live genes, fold 0~4
ard        ard      all sales   35, 41, 38, 38, 36
ridge      ridge    all sales   8, 10, 11, 10, 11
market     ridge    market sales9, 11, 10, 10, 13
nonmarket  ridge    nonmarket sales33, 31, 31, 42, 38


### Id sum check / Id 합 검증

In [9]:
# The fold table must be the competition train.csv minus the rows that contradict themselves.
# fold 표는 대회 train.csv에서 모순된 행만 뺀 것이어야 한다.
_extra = sorted(int(i) for i in set(train_csv.Id) - set(fold_tbl.Id))

print(f'{"item":<12}{"rows":>8}{"id sum":>14}')
for _k, _df in (('train.csv', train_csv), ('test.csv', test_csv), ('fold table', fold_tbl)):
    print(f'{_k:<12}{len(_df):>8,}{int(_df.Id.sum()):>14,}')

assert set(fold_tbl.Id) <= set(train_csv.Id), 'the fold table has ids that train.csv does not!!'
assert len(_extra) == 7, f'train.csv has {len(_extra)} rows outside the fold table, expected 7!!'
assert int(train_csv.Id.sum()) - int(fold_tbl.Id.sum()) == sum(_extra), 'id sum does not add up!!'

print(f'\n*** Id check passed ***')
print(f'    fold table = train.csv minus the 7 contradictory rows {_extra}')
print(f'    id sum gap {int(train_csv.Id.sum()) - int(fold_tbl.Id.sum()):,} = sum of those 7 ids')

item            rows        id sum
train.csv      1,460     1,066,530
test.csv       1,459     3,195,210
fold table     1,453     1,059,333

*** Id check passed ***
    fold table = train.csv minus the 7 contradictory rows [625, 689, 774, 1231, 1242, 1301, 1335]
    id sum gap 7,197 = sum of those 7 ids


# 3. Preprocessing / 전처리

원본 csv는 다섯 단계를 거쳐 설계행렬 89열이 되고, 그 위에서 두 가지를 더 만든다.

1. **모순 처리** : train 7행은 값끼리 맞지 않는다(지하실 유형은 '없음'인데 지하실 면적은 있는 식). 이 7행은 지우고, test의 6칸은 값을 바로잡는다.
2. **결측 채우기** : `NA`는 어떤 열에서는 '없음', 어떤 열에서는 '모름'이다. 두 경우를 다르게 채운다.
3. **범주형 인코딩** : 글자 열 43개를 세 갈래로 나눠 숫자로 바꾼다. 그중 `Utilities`는 뺀다.
4. **누출 통제** : 정답(가격)을 쓰는 통계는 모두 그 fold의 train 행만으로 계산한다. 이상치 두 채는 train 쪽에 두되, 가격 통계에서는 뺀다.
5. **조립** : 가공한 78열 + OOF 타깃 인코딩 11열 = 89열. 저장된 GP 유전자는 이 89열을 터미널로 쓴다.
6. **기본 피쳐** : 가공한 78열 + 원본 범주형 43열 = 121열. 전문가에게 들어가는 표의 바탕이다.
7. **판매 조건 축** : 행마다 시장 거래인지 비시장 거래인지 표시한다. 컨텍스트를 고르고 예측을 섞을 때 이 표시를 쓴다.

1\~5단계는 이전 노트북과 같다.<br>
저장된 GP 유전자를 같은 89열 위에서 계산해야 하기 때문이다.

## Rows: contradictions and missing values / 행: 모순과 결측

### Fix contradictions and fill missing / 모순 처리와 결측 채우기

In [10]:
# Stage 1-2: remove self-contradicting rows, then fill the two kinds of NA.
# 1~2단계: 값끼리 모순되는 행을 지우고, 뜻이 다른 두 종류의 NA를 각각 다르게 채운다.
#   NA_IS_NONE   : 'NA'가 '없음'이라는 뜻인 열 (지하실 없음, 차고 없음 ...)
#   FIXED_FILL   : 'NA'가 '모름'이라 정해 둔 값으로 채우는 열
#   PAIR         : 한쪽이 없으면 다른 쪽도 없어야 하는 짝 (지하실 유형 ↔ 면적)


# 'NA'만 결측으로 읽는다. MasVnrType의 'None'은 값이다.
NA_OPTS = dict(keep_default_na=False, na_values=['NA'])


# Seven training rows contradict themselves (a basement area with no basement type). They go out.
# train 데이터 중 7행은 값끼리 모순된다(지하실 유형은 없는데 면적은 있는 식). 이 행들은 지운다.
TRAIN_DROP_IDS = (625, 689, 774, 1231, 1242, 1301, 1335)

TEST_YEAR_FIX_IDS = (1877, 2550, 2593)

TEST_MASVNR_FIX_IDS = (1670, 2453, 2320)

BSMT_CATS = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

BSMT_NUMS = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath']

GARAGE_CATS = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']

GARAGE_NUMS = ['GarageYrBlt', 'GarageCars', 'GarageArea']

PAIR = {**{c: ('TotalBsmtSF', 'NA') for c in BSMT_CATS},
        **{c: ('GarageArea', 'NA') for c in GARAGE_CATS},
        'FireplaceQu': ('Fireplaces', 'NA'),
        'PoolQC': ('PoolArea', 'NA'),
        'MasVnrType': ('MasVnrArea', 'None')}


# 'NA' means "none" in these columns and "unknown" elsewhere. The two cases are filled differently.
# 이 열들에서 'NA'는 '없음'이고, 다른 열에서는 '모름'이다. 두 경우를 다르게 채운다.
NA_IS_NONE = ['Alley', 'Fence', 'MiscFeature']


# NA가 '모름'인 열과 그 열을 채울 값.
FIXED_FILL = {'MSZoning': 'RL', 'Utilities': 'AllPub', 'Exterior1st': 'VinylSd', 'Exterior2nd': 'VinylSd',
              'KitchenQual': 'TA', 'Functional': 'Typ', 'SaleType': 'WD', 'Electrical': 'SBrkr'}


# 면적이 0보다 큰데 유형이 NA인 행에 넣을 값.
MASVNR_TYPE_WITH_AREA = 'BrkFace'

def numeric_columns(df):
    return [c for c in df.columns if c not in ('Id', 'SalePrice') and pd.api.types.is_numeric_dtype(df[c])]


# Fix what can be fixed, drop what cannot. Test rows are never dropped, since the submission needs all 1,459.
# 고칠 수 있으면 고치고 못 고치면 버린다. test 행은 절대 버리지 않는다. 제출에 1,459행이 다 필요하다.
def fix_contradictions(train, test):
    train = train[~train['Id'].isin(TRAIN_DROP_IDS)].reset_index(drop=True)
    test = test.copy()
    changes = []

    def change(mask, col, new, rule):
        for idx in test.index[mask]:
            value = new[idx] if isinstance(new, pd.Series) else new
            if test.at[idx, col] != value:
                changes.append(dict(Id=int(test.at[idx, 'Id']), column=col, before=test.at[idx, col], after=value,
                                    rule=rule))
            test.at[idx, col] = value

    change(test['GarageYrBlt'] == 2207, 'GarageYrBlt', 2007, 'GarageYrBlt 2207 → 2007')
    change(test['YearRemodAdd'] < test['YearBuilt'], 'YearRemodAdd', test['YearBuilt'], 'YearRemodAdd < YearBuilt → YearBuilt')
    change(test['YrSold'] < test['YearBuilt'], 'YrSold', test['YearBuilt'], 'YrSold < YearBuilt → YearBuilt')
    area_one = test['MasVnrArea'] == 1
    change(area_one, 'MasVnrType', 'None', '마감재 면적 1 → 0·None')
    change(area_one, 'MasVnrArea', 0.0, '마감재 면적 1 → 0·None')
    change((test['MasVnrType'] == 'None') & (test['MasVnrArea'] > 1), 'MasVnrType', 'BrkFace', 'None + 면적 > 1 → BrkFace')
    has_type = test['MasVnrType'].notna() & (test['MasVnrType'] != 'None')
    change(has_type & (test['MasVnrArea'] == 0), 'MasVnrType', 'None', '종류 있음 + 면적 0 → None')
    return train, test, pd.DataFrame(changes, columns=['Id', 'column', 'before', 'after', 'rule'])

def _mode(values):
    modes = values.mode()
    return modes.iloc[0], list(modes)


# Paired columns decide whether an NA is "none" or "unknown": if the partner is 0 it is none, else unknown.
# 짝 컬럼으로 NA의 뜻을 가린다. 짝이 0이면 없음, 0보다 크면 모름(최빈값으로 채운다).
def fill_missing(train, test):
    pool = pd.concat([train, test], ignore_index=True)
    nb_lotfrontage = pool.groupby('Neighborhood')['LotFrontage'].median()
    fills, out = [], []

    for name, df in (('train', train), ('test', test)):
        df = df.copy()

        def fill(mask, col, value, rule):
            for idx in df.index[mask]:
                v = value[idx] if isinstance(value, pd.Series) else value
                fills.append(dict(dataset=name, Id=int(df.at[idx, 'Id']), column=col, value=v, rule=rule))
                df.at[idx, col] = v

        no_bsmt = df[BSMT_CATS].isna().all(axis=1) & (df['TotalBsmtSF'].isna() | (df['TotalBsmtSF'] == 0))
        for col in BSMT_NUMS:
            fill(no_bsmt & df[col].isna(), col, 0.0, '시설 없음 → 0')
        fill(df['GarageType'].isna() & df['GarageYrBlt'].isna(), 'GarageYrBlt', df['YearBuilt'], '차고 없음 → YearBuilt')
        for col in GARAGE_NUMS:
            fill(df['GarageType'].notna() & df[col].isna(), col, pool[col].median(), '차고 있음 → 중앙값')
        fill(df['LotFrontage'].isna(), 'LotFrontage', df['Neighborhood'].map(nb_lotfrontage), '지역별 중앙값')
        for col in numeric_columns(df):
            fill(df[col].isna(), col, pool[col].median(), '중앙값')

        for col, value in FIXED_FILL.items():
            fill(df[col].isna(), col, value, '정해 둔 값')
        for col in NA_IS_NONE:
            fill(df[col].isna(), col, 'NA', '없음')
        for col, (pair, none_value) in PAIR.items():
            fill(df[col].isna() & (df[pair] == 0), col, none_value, f'{pair} = 0 → 없음')
            missing = df[col].isna() & (df[pair] > 0)
            if not missing.any():
                continue
            if col == 'MasVnrType':
                fill(missing, col, MASVNR_TYPE_WITH_AREA, f'{pair} > 0 → BrkFace (면적이 있는데 유형이 비었다)')
                continue
            mode, modes = _mode(pool[col])
            tie = f' (동률 {"·".join(modes)} → {mode})' if len(modes) > 1 else ''
            fill(missing, col, mode, f'{pair} > 0 → 최빈값{tie}')
        out.append(df)

    fills = pd.DataFrame(fills, columns=['dataset', 'Id', 'column', 'value', 'rule'])
    return out[0], out[1], fills

## Columns: categorical encoding / 열: 범주형 인코딩

### Three routes for 43 string columns / 글자 열 43개의 세 갈래

범주형 피쳐에 명확한 순서가 있으면 그것을 사용하고, 순서가 없는 경우 OOF 타깃 인코딩을 사용한다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">갈래</th><th style="text-align:center">열 수</th><th style="text-align:left">이유</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>순서형 정수</b></td><td style="text-align:center">23</td><td style="text-align:left">누구나 납득할 만한 순서가 있다(<code>Ex &gt; Gd &gt; TA &gt; Fa &gt; Po</code>). 한 열이면 되고, 추정 잡음도 누출도 없다</td></tr>
<tr><td style="text-align:left"><b>파생 그룹</b></td><td style="text-align:center">9 → 21</td><td style="text-align:left">수준이 너무 많아서, 도메인 지식으로 묶는 편이 낫다(외장재 계열, 동네 평당가)</td></tr>
<tr><td style="text-align:left"><b>타깃 인코딩</b></td><td style="text-align:center">11</td><td style="text-align:left">납득할 만한 순서가 없으니, 정답(가격)으로 순서를 매긴다</td></tr>
</tbody></table>
</div>

순서형 23개까지 타깃 인코딩으로 바꾸면, 이미 아는 순서를 버리고 그 자리를 추정 잡음으로 채우는 셈이다.

In [11]:
# Stage 3: one route per column, chosen by what the column's own structure allows.
# 3단계: 열마다 그 열의 구조에 맞는 방식으로 인코딩한다.
#   ORDINAL   : 품질 등급을 정수 순서로 바꾼다(Ex=5 ... Po=1). Utilities도 여기서 인코딩하지만 설계행렬에서는 뺀다(build의 drop_cols)
#   DECOMPOSE : 한 열을 뜻이 다른 여러 0/1 열로 분해 (HouseStyle → 층수·반층·상층마감)
#   EXTERIOR  : 외장재 15종 → 재료 계열 5개 + '혼합' 표시
#   ONEHOT    : 순서가 없는 9열. 여기서 만든 0/1 열은 버리고 타깃 인코딩으로 대체한다(GarageType·MasVnrType까지 11열)
#   UNIT_PRICE: 동네별 평당가. 유일하게 가격을 쓰는 열이고, train 행에서는 자기 행을 빼고 계산한다

QUALITY = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}


# 23 columns have a defensible order, so they become integers. One column, no estimation noise, no leakage.
# 23개 열은 누구나 납득할 순서가 있어 정수로 바꾼다. 한 열로 충분하고 추정 잡음도 누출도 없다.
ORDINAL = {
    'ExterQual': dict(QUALITY),
    'ExterCond': dict(QUALITY),
    'BsmtQual': {'NA': 0, **QUALITY},
    'BsmtCond': {'NA': 0, **QUALITY},
    'BsmtExposure': {'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'HeatingQC': dict(QUALITY),
    'KitchenQual': dict(QUALITY),
    'FireplaceQu': {'NA': 0, **QUALITY},
    'GarageType': {'NA': 0, 'CarPort': 1, 'Detchd': 2, 'Attchd': 3, 'Basment': 3, '2Types': 3, 'BuiltIn': 4},
    'GarageFinish': {'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'GarageQual': {'NA': 0, **QUALITY},
    'GarageCond': {'NA': 0, **QUALITY},
    'PoolQC': {'NA': 0, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'MasVnrType': {'None': 0, 'BrkCmn': 1, 'BrkFace': 2, 'Stone': 3},
    'Alley': {'NA': 0, 'Grvl': 1, 'Pave': 2},
    'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
    'LotShape': {'Reg': 1, 'IR1': 2, 'IR2': 3, 'IR3': 4},
    'LandSlope': {'Gtl': 1, 'Mod': 2, 'Sev': 3},
    'PavedDrive': {'N': 1, 'P': 2, 'Y': 3},
    'Electrical': {'FuseP': 1, 'FuseF': 2, 'FuseA': 3, 'SBrkr': 4, 'Mix': 2},
    'Utilities': {'ELO': 1, 'NoSeWa': 2, 'NoSewr': 3, 'AllPub': 4},
    'OverallQual': {i: i for i in range(1, 11)},
    'OverallCond': {i: i for i in range(1, 11)},
}

NONE_CODES = ('NA', 'None')

BINARY = {
    'CentralAir': {'N': 0, 'Y': 1},
    'Street': {'Grvl': 0, 'Pave': 1},
}

_HOUSE_STYLES = ['1Story', '1.5Fin', '1.5Unf', '2Story', '2.5Fin', '2.5Unf', 'SFoyer', 'SLvl']

_BLDG_TYPES = ['1Fam', '2fmCon', 'Duplex', 'TwnhsE', 'Twnhs']

_LOT_CONFIGS = ['Inside', 'Corner', 'CulDSac', 'FR2', 'FR3']

_CONDITIONS = ['Norm', 'Feedr', 'Artery', 'RRNn', 'RRNe', 'RRAn', 'RRAe', 'PosN', 'PosA']

_FENCES = ['NA', 'GdPrv', 'MnPrv', 'GdWo', 'MnWw']

def _rest(codes, value, **explicit):
    return {c: explicit.get(c, value) for c in codes}


# 뜻이 섞인 열을 속성별 0/1 열로 분해한다. HouseStyle → 층수·반층·상층마감.
DECOMPOSE = {
    'HouseStyle': {
        'HouseStyle_Floors': ('층수', {'1Story': 1, '1.5Fin': 1.5, '1.5Unf': 1.5, '2Story': 2, '2.5Fin': 2.5,
                                      '2.5Unf': 2.5, 'SFoyer': 1.5, 'SLvl': 1.5}),
        'HouseStyle_UpperFin': ('위층 마감', _rest(_HOUSE_STYLES, 1, **{'1.5Unf': 0, '2.5Unf': 0})),
        'HouseStyle_Split': ('스플릿', _rest(_HOUSE_STYLES, 0, SFoyer=1, SLvl=1)),
    },
    'BldgType': {
        'BldgType_Units': ('가구 수', {'1Fam': 1, 'TwnhsE': 1, 'Twnhs': 1, 'Duplex': 2, '2fmCon': 2}),
        'BldgType_SharedWalls': ('타운하우스 공유 벽 수', {'1Fam': 0, '2fmCon': 0, 'Duplex': 0, 'TwnhsE': 1, 'Twnhs': 2}),
        'BldgType_Conversion': ('개조', _rest(_BLDG_TYPES, 0, **{'2fmCon': 1})),
    },
    'LotConfig': {
        'LotConfig_Frontages': ('도로 접면 수', {'Inside': 1, 'CulDSac': 1, 'Corner': 2, 'FR2': 2, 'FR3': 3}),
        'LotConfig_Corner': ('코너', _rest(_LOT_CONFIGS, 0, Corner=1)),
        'LotConfig_CulDSac': ('막다른 길', _rest(_LOT_CONFIGS, 0, CulDSac=1)),
    },
    'Condition': {
        'Condition_Road': ('큰 도로', _rest(_CONDITIONS, 0, Feedr=1, Artery=2)),
        'Condition_Rail': ('철도', _rest(_CONDITIONS, 0, RRNn=1, RRNe=1, RRAn=2, RRAe=2)),
        'Condition_Pos': ('긍정 요소', _rest(_CONDITIONS, 0, PosN=1, PosA=2)),
    },
    'Fence': {
        'Fence_Privacy': ('프라이버시 울타리', _rest(_FENCES, 0, MnPrv=1, GdPrv=2)),
        'Fence_Wood': ('목재 울타리', _rest(_FENCES, 0, MnWw=1, GdWo=2)),
    },
}

DECOMPOSE_SOURCES = {'HouseStyle': ['HouseStyle'], 'BldgType': ['BldgType'], 'LotConfig': ['LotConfig'],
                     'Condition': ['Condition1', 'Condition2'], 'Fence': ['Fence']}


# 외장재 15종을 재료 계열 5개로 묶는다. 비닐이 기준이고 두 면이 다르면 혼합 표시를 켠다.
EXTERIOR_FAMILY = {
    'VinylSd': 'Vinyl',
    'MetalSd': 'Metal',
    'Wd Sdng': 'Wood', 'WdShing': 'Wood', 'Plywood': 'Wood', 'HdBoard': 'Wood',
    'CemntBd': 'CementStucco', 'Stucco': 'CementStucco', 'ImStucc': 'CementStucco',
    'BrkFace': 'BrickStone', 'BrkComm': 'BrickStone', 'Stone': 'BrickStone', 'CBlock': 'BrickStone',
    'AsbShng': 'ShingleOther', 'AsphShn': 'ShingleOther', 'Other': 'ShingleOther', 'PreCast': 'ShingleOther',
}

EXTERIOR_BASE = 'Vinyl'

EXTERIOR_COLUMNS = {
    'Exterior_Metal': ('금속', 'Metal'),
    'Exterior_Wood': ('목재', 'Wood'),
    'Exterior_CementStucco': ('시멘트·스터코', 'CementStucco'),
    'Exterior_BrickStone': ('벽돌·석재', 'BrickStone'),
    'Exterior_ShingleOther': ('슁글·기타', 'ShingleOther'),
}

EXTERIOR_MIXED = ('Exterior_Mixed', '외장재 혼합')

EXTERIOR2_SPELLING = {'CmentBd': 'CemntBd', 'Wd Shng': 'WdShing', 'Brk Cmn': 'BrkComm'}


# These 9 columns have no defensible order. The 0/1 columns made here are dropped and target-encoded instead.
# 이 9개 열은 순서가 없다. 여기서 만든 0/1 열은 버리고 타깃 인코딩으로 대체한다(아래 FACTOR_COLUMNS).
ONEHOT = {
    'MSZoning': [('MSZoning_RM', 'RM', ['RM']), ('MSZoning_RH', 'RH', ['RH']), ('MSZoning_FV', 'FV', ['FV']),
                 ('MSZoning_C', '상업', ['C (all)'])],
    'LandContour': [('LandContour_Bnk', '둔덕', ['Bnk']), ('LandContour_HLS', '언덕 사면', ['HLS']),
                    ('LandContour_Low', '저지대', ['Low'])],
    'RoofStyle': [('RoofStyle_Hip', '모임지붕', ['Hip']),
                  ('RoofStyle_Other', '기타 지붕 형태', ['Flat', 'Gambrel', 'Mansard', 'Shed'])],
    'RoofMatl': [('RoofMatl_Wood', '목재 지붕', ['WdShngl', 'WdShake']),
                 ('RoofMatl_Other', '기타 지붕 재료', ['Tar&Grv', 'Metal', 'Membran', 'Roll', 'ClyTile'])],
    'Foundation': [('Foundation_PConc', '타설 콘크리트', ['PConc']), ('Foundation_CBlock', '콘크리트 블록', ['CBlock']),
                   ('Foundation_BrkTil', '벽돌·타일', ['BrkTil'])],
    'Heating': [('Heating_HotWater', '온수·증기', ['GasW', 'OthW']),
                ('Heating_OldFurnace', '구식 난로', ['Grav', 'Wall', 'Floor'])],
    'MiscFeature': [('MiscFeature_Shed', '창고', ['Shed']),
                    ('MiscFeature_Other', '기타 시설', ['Gar2', 'Othr', 'TenC', 'Elev'])],
    'SaleType': [('SaleType_New', '신축 판매', ['New']), ('SaleType_Contract', '계약 판매', ['ConLD', 'ConLI', 'ConLw', 'Con']),
                 ('SaleType_COD', '법원·유산', ['COD'])],
    'SaleCondition': [('SaleCondition_Partial', '미완공 신축', ['Partial']),
                      ('SaleCondition_Abnorml', '비정상 거래', ['Abnorml']),
                      ('SaleCondition_Special', '특수 거래', ['Family', 'Alloca', 'AdjLand'])],
}

ONEHOT_BASE = {'MSZoning': ['RL'], 'LandContour': ['Lvl'], 'RoofStyle': ['Gable'], 'RoofMatl': ['CompShg'],
               'Foundation': ['Slab', 'Stone', 'Wood'], 'Heating': ['GasA'], 'MiscFeature': ['NA'],
               'SaleType': ['WD', 'CWD', 'VWD', 'Oth'], 'SaleCondition': ['Normal']}


# The only column that uses the price. Per-neighborhood unit price, and the training row's own value is left out.
# 가격을 쓰는 유일한 열. 동네별 평당가이고, train 행에서는 자기 행을 뺀 값을 쓴다.
UNIT_PRICE = 'Neighborhood_UnitPrice'


# MSSubClass는 HouseStyle + BldgType + 건축연도로 98.4%가 복원되므로 지운다.
DROP = ['MSSubClass']

NUMERIC = ['LotFrontage', 'LotArea', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
           'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
           'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars',
           'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
           'MiscVal', 'MoSold', 'YrSold']

CATEGORICAL_FEATURES = (list(ORDINAL) + list(BINARY)
                        + [col for group in DECOMPOSE.values() for col in group]
                        + list(EXTERIOR_COLUMNS) + [EXTERIOR_MIXED[0]]
                        + [col for cols in ONEHOT.values() for col, _, _ in cols]
                        + [UNIT_PRICE])

FEATURES = CATEGORICAL_FEATURES + NUMERIC

FIXED_FEATURES = [f for f in FEATURES if f != UNIT_PRICE]

SOURCE_COLUMNS = (list(ORDINAL) + list(BINARY) + [c for cols in DECOMPOSE_SOURCES.values() for c in cols]
                  + ['Exterior1st', 'Exterior2nd'] + list(ONEHOT) + ['Neighborhood'] + DROP)

def _map(series, mapping, name, numeric=True):
    unknown = set(series.dropna().unique()) - set(mapping)
    if unknown:
        raise ValueError(f'{name}: 매핑에 없는 값 {sorted(map(str, unknown))}')
    mapped = series.map(mapping)
    return mapped.astype(float) if numeric else mapped


# 가격을 쓰지 않는 인코딩 열과 수치형 열을 만든다. Id·Neighborhood는 함께 넘긴다.
def encode_fixed(df):
    cols = {}
    for col, mapping in {**ORDINAL, **BINARY}.items():
        cols[col] = _map(df[col], mapping, col)
    for group, attrs in DECOMPOSE.items():
        sources = DECOMPOSE_SOURCES[group]
        for new, (_, mapping) in attrs.items():
            values = [_map(df[s], mapping, f'{s}→{new}') for s in sources]
            cols[new] = values[0] if len(values) == 1 else np.maximum(*values)
    family1 = _map(df['Exterior1st'], EXTERIOR_FAMILY, 'Exterior1st', numeric=False)
    family2 = _map(df['Exterior2nd'].replace(EXTERIOR2_SPELLING), EXTERIOR_FAMILY, 'Exterior2nd', numeric=False)
    for new, (_, family) in EXTERIOR_COLUMNS.items():
        cols[new] = (family1 == family).astype(float)
    cols[EXTERIOR_MIXED[0]] = (family1 != family2).astype(float)
    for col, groups in ONEHOT.items():
        known = set(ONEHOT_BASE[col]) | {v for _, _, members in groups for v in members}
        unknown = set(df[col].dropna().unique()) - known
        if unknown:
            raise ValueError(f'{col}: 매핑에 없는 값 {sorted(unknown)}')
        for new, _, members in groups:
            cols[new] = df[col].isin(members).astype(float).where(df[col].notna())
    for col in NUMERIC:
        cols[col] = df[col].astype(float)
    keep = [c for c in ('Id', 'Neighborhood', 'SalePrice') if c in df.columns]
    out = pd.concat([df[keep], pd.DataFrame(cols, index=df.index)], axis=1)
    return out[keep + FIXED_FEATURES]

### `Utilities` is dropped / `Utilities`는 뺀다

train 1,453행 중 1,452행이 `AllPub`(전기 · 가스 · 수도 · 하수 모두 있음)이고, Id 945 한 채만 `NoSeWa`다.<br>
test는 모두 `AllPub`이거나 빈칸이고, 빈칸도 `AllPub`으로 채운다.<br>
이 열로 배울 수 있는 것은 그 한 채를 짚어 내는 법뿐이다.<br>
그래서 설계행렬(90 → 89열)에서도, 기본 피쳐의 원본 범주형에서도 뺀다.

인코딩은 그대로 하고, 설계행렬을 만들 때만 뺀다(`build(drop_cols=...)`).

## Folds: leakage control and the design matrix / fold별 누출 통제와 설계행렬

### Leakage control / 누출 통제

In [12]:
# Stage 4: leakage control. Every answer-touching statistic is fitted on the fold's train rows only.
# 4단계: 누출 통제. 정답(가격)을 쓰는 통계는 그 fold의 train 행만으로 계산한다.
#   FoldPreprocessor.fit(train)      : 동네 평당가·스케일 통계를 train 행에서 구한다. 이상치 두 채는 평당가 통계에서 뺀다
#   FoldPreprocessor.transform(other): valid·test 행은 가격을 **읽지 않는다**
#   make_folds                       : fold 배정. 이 노트북은 공유된 fold 표를 쓰고, 그 표가 이 함수의 결과와 같은지 확인한다

N_FOLDS = 5


# 타깃 인코딩의 평활 강도. 행이 적은 수준일수록 전체 평균 쪽으로 당겨진다.
SMOOTHING_M = 10


# 0에 뜻이 없고 값이 0에서 멀리 떨어진 열만 z-score한다. 나머지는 표준편차로만 나눈다.
ZSCORE = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'YrSold', 'MoSold', 'LotFrontage']

def scale_method(feature):
    return 'zscore' if feature in ZSCORE else 'divide'


# fit() sees the fold's training rows only; transform() never reads a price. This is the leakage boundary.
# fit()은 그 fold의 train 행만 본다. transform()은 가격을 읽지 않는다. 누출을 막는 경계가 바로 여기다.
# exclude_ids(이상치 두 채)는 동네 평균 단가 통계에서 빠지고, 두 채 자신은 valid 행처럼 나머지 행의 통계로 값을 받는다.
class FoldPreprocessor:

    def __init__(self, exclude_ids=()):
        self.exclude_ids = tuple(exclude_ids)

    def fit(self, enc):
        price = enc['SalePrice'].to_numpy(float)
        unit = price / enc['GrLivArea'].to_numpy(float)
        ex = enc['Id'].isin(self.exclude_ids).to_numpy()
        nb = enc['Neighborhood'].to_numpy()
        groups = pd.Series(unit[~ex]).groupby(nb[~ex])
        self.unit_mean_ = float(unit[~ex].mean())
        self.unit_sum_ = groups.sum().to_dict()
        self.unit_count_ = groups.count().to_dict()
        S, n = self._neighborhood_stats(enc)
        m, mu = SMOOTHING_M, self.unit_mean_
        loo = np.where(ex, (S + m * mu) / (n + m), (S - unit + m * mu) / (n - 1 + m))
        raw = self._assemble(enc, loo)
        self.mean_ = raw.mean(axis=0)
        std = raw.std(axis=0)
        self.std_ = np.where(std > 0, std, 1.0)
        self.method_ = np.array([scale_method(f) for f in FEATURES])
        self.center_ = np.where(self.method_ == 'zscore', self.mean_, 0.0)
        self.X_train_ = (raw - self.center_) / self.std_
        return self

    def transform(self, enc):
        S, n = self._neighborhood_stats(enc)
        m, mu = SMOOTHING_M, self.unit_mean_
        return (self._assemble(enc, (S + m * mu) / (n + m)) - self.center_) / self.std_

    def _neighborhood_stats(self, enc):
        nb = enc['Neighborhood']
        S = nb.map(self.unit_sum_).fillna(0.0).to_numpy(float)
        n = nb.map(self.unit_count_).fillna(0).to_numpy(float)
        return S, n

    @staticmethod
    def _assemble(enc, unit_price):
        raw = pd.DataFrame(enc[FIXED_FEATURES].to_numpy(float), columns=FIXED_FEATURES)
        raw[UNIT_PRICE] = unit_price
        raw = raw[FEATURES].to_numpy(float)
        if np.isnan(raw).any():
            bad = sorted({FEATURES[j] for j in np.where(np.isnan(raw))[1]})
            raise ValueError(f'재료에 빈 칸이 있다: {bad}')
        return raw


# fold 배정. 이 노트북은 공유된 fold 표를 쓰고, 그 표가 이 함수의 결과와 같은지 확인한다.
def make_folds(n_rows, n_folds=N_FOLDS, seed=SEED):
    fold = np.empty(n_rows, dtype=int)
    for k, (_, valid_idx) in enumerate(KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(np.arange(n_rows))):
        fold[valid_idx] = k
    return fold

@dataclass
class FoldData:
    X_train: np.ndarray
    y_train: np.ndarray
    X_valid: np.ndarray
    y_valid: np.ndarray
    ids_train: np.ndarray
    ids_valid: np.ndarray


# fold k를 valid로 두고 나머지 행으로 전처리를 적합한다.
def fold_data(enc, folds, k, exclude_ids):
    train, valid = enc[folds != k].reset_index(drop=True), enc[folds == k].reset_index(drop=True)
    pre = FoldPreprocessor(exclude_ids).fit(train)
    return FoldData(pre.X_train_, np.log(train['SalePrice'].to_numpy(float)), pre.transform(valid),
                    np.log(valid['SalePrice'].to_numpy(float)), train['Id'].to_numpy(), valid['Id'].to_numpy()), pre

### Two outliers stay on the train side / 이상치 두 채는 train 쪽에 둔다

Id 524와 1299는 4,676 · 5,642sqft의 큰 집인데, 신축 분양(`Partial`) 거래로 184,750 · 160,000달러에 팔렸다.<br>
데이터셋 저자도 이런 거래는 시장 가치를 반영하지 않을 가능성이 크다고 적었다(De Cock 2011).<br>
그래도 train에서 빼지 않는다.<br>
test에도 같은 유형의 집(Id 2550, 5,095sqft 분양)이 있고, 큰 집이 싸게 팔리는 경우를 보여 주는 행이 이 두 채뿐이기 때문이다.

`build`가 두 가지를 한다.

- `pin_ids` : 두 채의 fold 번호를 −1로 바꿔 어느 fold에서도 valid 행이 되지 않게 한다. 다른 행의 fold 번호와 OOF용 내부 fold 번호는 그대로다. 이 분할에서는 fold 1 · 2의 valid 행이 291행에서 290행이 된다.
- `stat_exclude_ids` : 타깃 인코딩의 수준별 평균과 동네 평균 단가를 낼 때 두 채를 뺀다. 두 채 자신의 값은 나머지 행의 통계로 채운다.

두 채는 `Partial` 거래라 시장 거래 전문가의 컨텍스트에 늘 들어간다.

### Out-of-fold target encoding / OOF 타깃 인코딩

타깃 인코딩은 범주의 각 수준을 그 수준에 속한 행들의 평균 정답으로 바꾼다.<br>
그대로 쓰면 각 행의 정답이 그 행의 피쳐에 섞여 들어간다.

OOF는 train 행을 다시 내부 5 fold로 나누고, 각 행에 나머지 네 fold로 낸 통계를 준다.<br>
자기 정답이 자기 피쳐에 들어가지 않으니, train 행도 valid 행과 같은 조건이 된다.<br>
valid · test 행은 train 행 전체의 통계를 쓴다.

이 11열은 저장된 GP 유전자의 터미널로만 쓰고 기본 피쳐에는 넣지 않는다.<br>
기본 피쳐에는 같은 열의 원래 값이 범주로 이미 들어 있다.

In [13]:
# Stage 5: the design matrix. 78 engineered + 11 out-of-fold target encodings = 89 columns, the terminals of the saved genes.
# 5단계: 설계행렬. 가공 78열 + OOF 타깃 인코딩 11열 = 89열이고, 저장된 유전자가 이 89열을 터미널로 쓴다.
#   FACTOR_COLUMNS   : 순서가 없어 타깃 인코딩으로 보내는 11열
#   KEPT             : 그 11열의 인코딩을 뺀 나머지 79열. build가 drop_cols(Utilities)를 빼서 78열이 된다
#   target_encode_oof: OOF 타깃 인코딩. 위에서 설명한 방식을 구현한 함수다
#   build            : fold 하나의 Design(train·valid·test 행렬과 Id)을 만든다. 이상치 두 채는 pin_ids로 train 쪽에 둔다

OOF_FOLDS = 5

FACTOR_ONEHOT = list(ONEHOT)

FACTOR_ORDINAL = ['GarageType', 'MasVnrType']


# The 11 nominal columns with no defensible order. These get the out-of-fold target encoding.
# 순서가 없는 명목형 11열. 이 열들은 OOF 타깃 인코딩으로 바꾼다.
FACTOR_COLUMNS = FACTOR_ONEHOT + FACTOR_ORDINAL

DROPPED = [new for col in FACTOR_ONEHOT for new, _, _ in ONEHOT[col]] + FACTOR_ORDINAL


# 재료 79열. 위 11열의 인코딩을 뺀 나머지다. build가 drop_cols(Utilities)를 빼서 터미널은 78 + 11 = 89개다.
KEPT = [f for f in FEATURES if f not in set(DROPPED)]


# 컬럼의 수준 목록. 드문 값 병합은 원래 인코딩 규칙을 그대로 따른다.
def factor_levels(col):
    if col in ONEHOT:
        groups = [list(ONEHOT_BASE[col])] + [list(vals) for _, _, vals in ONEHOT[col]]
        return groups
    mapping = ORDINAL[col]
    codes = sorted(set(mapping.values()))
    return [[k for k, v in mapping.items() if v == code] for code in codes]

def level_index(col):
    return {v: i for i, group in enumerate(factor_levels(col)) for v in group}


# Out-of-fold: a training row sees the statistic of the four inner folds it does not belong to.
# OOF. train 행은 자기가 속하지 않은 내부 네 fold의 통계를 본다.
# 자기 정답이 자기 피쳐에 들어가지 않으므로 역산할 대상이 없다. valid·test 행은 train 전체 통계를 쓴다.
# exclude가 True인 행(이상치 두 채)은 어느 통계에도 넣지 않는다. 그 행도 값은 받는다.
def target_encode_oof(codes_tr, y_tr, n_lv, inner, codes_other=None, m=SMOOTHING_M, exclude=None):
    keep = np.ones(len(y_tr), bool) if exclude is None else ~np.asarray(exclude, bool)
    tr = np.empty(len(codes_tr), float)
    for j in np.unique(inner):
        ref = (inner != j) & keep
        y_ref, c_ref = y_tr[ref], codes_tr[ref]
        mu_j = float(y_ref.mean())
        S_j = np.bincount(c_ref, weights=y_ref, minlength=n_lv)
        n_j = np.bincount(c_ref, minlength=n_lv).astype(float)
        lvl_j = (S_j + m * mu_j) / (n_j + m)
        tr[inner == j] = lvl_j[codes_tr[inner == j]]
    mu = float(y_tr[keep].mean())
    S = np.bincount(codes_tr[keep], weights=y_tr[keep], minlength=n_lv)
    n = np.bincount(codes_tr[keep], minlength=n_lv).astype(float)
    lvl = (S + m * mu) / (n + m)
    return tr, (lvl[codes_other] if codes_other is not None else None), lvl


# Pinning moves two rows into a fold's training side; every other row keeps its inner fold number.
# 고정 때문에 train 쪽으로 새로 들어온 행만 내부 fold Id % 5번에 넣는다. 나머지 행의 내부 fold 번호는 그대로다.
def _inner_folds(n_train, new_rows, ids_train):
    if not new_rows.any():
        return make_folds(n_train, n_folds=OOF_FOLDS)
    inner = np.empty(n_train, int)
    inner[~new_rows] = make_folds(int((~new_rows).sum()), n_folds=OOF_FOLDS)
    inner[new_rows] = np.asarray(ids_train)[new_rows] % OOF_FOLDS
    return inner


# fold 하나의 재료. train·valid·test 행렬과 Id, 그리고 열 이름이 들어 있다.
@dataclass
class Design:
    fold: int
    X_train: np.ndarray
    y_train: np.ndarray
    X_valid: np.ndarray
    y_valid: np.ndarray
    X_test: np.ndarray
    features: list
    ids_train: np.ndarray
    ids_valid: np.ndarray
    ids_test: np.ndarray

def _codes(raw, cols):
    out = {}
    for col in cols:
        idx = level_index(col)
        src = raw['Condition1'] if col == 'Condition' else raw[col]
        out[col] = src.map(idx).to_numpy(np.int64)
        assert not np.any(out[col] < 0), col
    return out


# One fold's design matrix: 78 engineered columns + 11 out-of-fold target encodings = 89 columns.
# fold 하나의 설계행렬. 가공 78열 + OOF 타깃 인코딩 11열 = 89열.
#   drop_cols        : 설계행렬에서 뺄 열(Utilities). 인코딩은 그대로 하고 설계행렬에서만 뺀다
#   pin_ids          : 모든 fold에서 train 쪽에 둘 Id. fold 번호를 -1로 두어 어느 fold의 valid에도 들어가지 않는다
#   stat_exclude_ids : 가격에서 나온 통계(타깃 인코딩 수준 평균, 동네 평균 단가)에서 뺄 Id
def build(fold, folds, prepared, drop_cols=(), pin_ids=(), stat_exclude_ids=()):
    unknown = set(drop_cols) - set(KEPT)
    if unknown:
        raise ValueError(f'drop_cols에 재료에 없는 열이 있다: {sorted(unknown)}')
    kept = [f for f in KEPT if f not in set(drop_cols)]
    ts = prepared.train
    enc_all = encode_fixed(ts)
    keep = [FEATURES.index(f) for f in kept]
    f0 = np.asarray(folds)
    f = f0.copy()
    f[ts['Id'].isin(list(pin_ids)).to_numpy()] = -1
    fd, pre = fold_data(enc_all, f, fold, stat_exclude_ids)
    tr_raw = ts[f != fold].reset_index(drop=True)
    va_raw = ts[f == fold].reset_index(drop=True)
    new_rows = f0[f != fold] == fold
    Xtr = np.ascontiguousarray(fd.X_train[:, keep])
    Xva = np.ascontiguousarray(fd.X_valid[:, keep])
    codes_tr = _codes(tr_raw, FACTOR_COLUMNS)
    codes_va = _codes(va_raw, FACTOR_COLUMNS)
    n_lv = {c: len(factor_levels(c)) for c in FACTOR_COLUMNS}
    names = list(kept)

    # 타깃 인코딩 11열. train 행은 OOF 값, valid·test 행은 train 전체의 수준 평균이고, train 행 기준으로 표준화한다
    inner = _inner_folds(len(fd.y_train), new_rows, fd.ids_train)
    excl = np.isin(fd.ids_train, list(stat_exclude_ids)) if stat_exclude_ids else None
    extra_tr, extra_va, levels = [], [], {}
    for c in FACTOR_COLUMNS:
        t, v, lvl = target_encode_oof(codes_tr[c], fd.y_train, n_lv[c], inner, codes_va[c], exclude=excl)
        mu, sd = float(t.mean()), float(t.std())
        sd = sd if sd > 1e-12 else 1.0
        extra_tr.append((t - mu) / sd)
        extra_va.append((v - mu) / sd)
        levels[c] = (lvl - mu) / sd
        names.append(f'{c}_TE')
    Xtr = np.ascontiguousarray(np.column_stack([Xtr] + extra_tr))
    Xva = np.ascontiguousarray(np.column_stack([Xva] + extra_va))

    # test 행: 같은 전처리, 같은 수준 평균
    test = prepared.test
    Xte = np.ascontiguousarray(pre.transform(encode_fixed(test.assign(SalePrice=1.0)))[:, keep])
    cd = _codes(test, FACTOR_COLUMNS)
    Xte = np.ascontiguousarray(np.column_stack([Xte] + [levels[c][cd[c]] for c in FACTOR_COLUMNS]))

    return Design(fold=fold, X_train=Xtr, y_train=fd.y_train, X_valid=Xva, y_valid=fd.y_valid, X_test=Xte,
                  features=names, ids_train=fd.ids_train, ids_valid=fd.ids_valid, ids_test=test['Id'].to_numpy())

### Build the design matrices / 설계행렬 만들기

In [14]:
# Every usable training row goes into the five folds. Nothing is held out.
# 쓸 수 있는 train 행을 모두 fold 다섯 개에 나눠 넣는다. 따로 떼어 두는 행은 없다.
train_fixed, test_fixed, CHANGES = fix_contradictions(train_csv, test_csv)
train_fixed, test_fixed, FILLS = fill_missing(train_fixed, test_fixed)

PREP = SimpleNamespace(train=train_fixed, test=test_fixed)
FOLDS = train_fixed.Id.map(FOLD_OF).to_numpy()
assert not pd.isna(FOLDS).any(), 'a training row has no fold number!!'
FOLDS = FOLDS.astype(int)

DES = {f: build(f, FOLDS, PREP, drop_cols=DROP_COLS, pin_ids=PIN_IDS, stat_exclude_ids=STAT_EXCLUDE_IDS)
       for f in range(5)}
print(f'rows fixed {len(train_fixed):,} · value changes {len(CHANGES)} · fills {len(FILLS)}')
for f in range(5):
    d = DES[f]
    print(f'  fold {f}: train {len(d.y_train):,} · valid {len(d.y_valid):,} '
          f'· columns {d.X_train.shape[1]} · test {len(d.ids_test):,}')

rows fixed 1,453 · value changes 6 · fills 13933
  fold 0: train 1,162 · valid 291 · columns 89 · test 1,459
  fold 1: train 1,163 · valid 290 · columns 89 · test 1,459
  fold 2: train 1,163 · valid 290 · columns 89 · test 1,459
  fold 3: train 1,163 · valid 290 · columns 89 · test 1,459
  fold 4: train 1,163 · valid 290 · columns 89 · test 1,459


### Preprocessing sanity check / 전처리 정합성 검사

In [15]:
# Check 1 of 4: the data. Every assert here is against the data itself, not a number measured earlier.
# 검증 1/4: 데이터. 여기 있는 assert는 모두 바깥 기록이 아니라 데이터 자체를 근거로 한다.
_dropped = sorted(int(i) for i in set(train_csv.Id) - set(train_fixed.Id))
assert len(_dropped) == 7, f'{len(_dropped)} rows were dropped, expected 7: {_dropped}!!'
assert set(train_fixed.Id) == set(fold_tbl.Id), 'the fixed training rows and the fold table differ!!'

# The shared fold table has to be what make_folds produces here. A sklearn version difference would show up here,
# and missing it would change every fold silently.
# 공유된 fold 표는 이 환경에서 make_folds를 돌린 결과와 같아야 한다. sklearn 버전 차이가 있으면 여기서 드러나고,
# 놓치면 모든 fold가 소리 없이 달라진다.
assert np.array_equal(make_folds(len(train_fixed)), FOLDS), \
    'make_folds disagrees with the shared fold table (sklearn version drift?)!!'

print(f'contradiction rows removed {_dropped}')
print(f'fold sizes {np.bincount(FOLDS).tolist()} · make_folds agrees with the shared table')

print(f'\n{"fold":>5}{"train":>7}{"valid":>7}{"test":>7}{"columns":>9}{"y sum":>13}')
for f in range(5):
    d = DES[f]
    assert d.X_train.shape[1] == 89, f'fold {f} has {d.X_train.shape[1]} columns, not 89!!'
    assert not set(DROP_COLS) & set(d.features), f'fold {f} still has {DROP_COLS} as a column!!'
    assert len(d.y_train) + len(d.y_valid) == len(train_fixed), f'fold {f} rows do not add up!!'
    # The two outliers are training rows in every fold, never validation rows.
    # 이상치 두 채는 모든 fold에서 train 행이고, 어느 fold에서도 valid 행이 아니다.
    assert set(PIN_IDS) <= set(d.ids_train) and not set(PIN_IDS) & set(d.ids_valid), \
        f'fold {f}: {PIN_IDS} are not pinned to the train side!!'
    assert np.isfinite(d.X_train).all() and np.isfinite(d.X_valid).all(), f'fold {f} non-finite!!'
    assert np.isfinite(d.X_test).all(), f'fold {f} test matrix has non-finite values!!'
    assert not (set(d.ids_train) & set(d.ids_valid)), f'fold {f} train and valid share a row!!'
    print(f'{f:>5}{len(d.y_train):>7,}{len(d.y_valid):>7,}{len(d.ids_test):>7,}'
          f'{d.X_train.shape[1]:>9}{float(d.y_train.sum()):>13.4f}')

print('\n*** Preprocessing check passed: rows, folds, pins, columns and finiteness all hold ***')

contradiction rows removed [625, 689, 774, 1231, 1242, 1301, 1335]
fold sizes [291, 291, 291, 290, 290] · make_folds agrees with the shared table

 fold  train  valid   test  columns        y sum
    0  1,162    291  1,459       89   13981.3067
    1  1,163    290  1,459       89   13991.0307
    2  1,163    290  1,459       89   13976.5664
    3  1,163    290  1,459       89   13976.0928
    4  1,163    290  1,459       89   13978.8537

*** Preprocessing check passed: rows, folds, pins, columns and finiteness all hold ***


## Base features and the sale-condition axis / 기본 피쳐 121열과 판매 조건 축

- **수치 78열** : 설계행렬 89열의 앞 78열이다. 순서형 정수, 0/1, 속성 분해, 원래 수치, 동네 평균 단가가 표준화된 채로 들어간다.
- **원본 범주형 43열** : 대회 데이터의 글자 열 42개(`Utilities` 제외)와, 숫자로 적힌 범주 `MSSubClass`다. 열 이름 앞에 `RAW_`를 붙이고, 모순 처리와 결측 채우기를 거친 값을 글자 그대로 둔다(예: 골목이 없으면 `NA`).
  - `MSSubClass`는 설계행렬에서는 뺐지만(주택 양식 · 건물 유형 · 건축 연도로 98.4%가 복원된다), 기본 피쳐에는 원래 값을 범주로 넣는다.
- **판매 조건 축** : `SaleCondition`이 `Normal` · `Partial`이면 시장 거래, `Abnorml` · `Family` · `Alloca` · `AdjLand`면 비시장 거래다.

열은 수치 78열, 원본 범주형 43열 순으로 놓고, 행 순서는 설계행렬과 같다.

In [16]:
# The 121 base features: the 78 numeric design columns (the 89 minus the 11 target encodings) and 43 raw labels.
# 기본 피쳐 121열: 설계행렬의 수치 78열(89열에서 타깃 인코딩 11열을 뺀 것)과 원본 범주형 43열.
#   RAW_COLS   : 대회 데이터의 글자 열 42개(Utilities 제외)와 숫자로 적힌 범주 MSSubClass. 값은 모순 처리 · 결측 채우기를 거친 글자다
#   sale_group : 행마다 시장 거래(market)인지 비시장 거래(nonmarket)인지 표시한다
N_NUM = 78
RAW_COLS = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope', 'Neighborhood',
            'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd',
            'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
            'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu',
            'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
            'SaleType', 'SaleCondition', 'MSSubClass']
RAW_NAMES = [f'RAW_{c}' for c in RAW_COLS]

# The raw labels are every text column of the competition data except Utilities, in file order, plus MSSubClass.
# 원본 범주형은 대회 데이터의 글자 열 전부(Utilities 제외, 파일 순서)와 MSSubClass다.
_text = [c for c in train_fixed.columns if not pd.api.types.is_numeric_dtype(train_fixed[c]) and c not in DROP_COLS]
assert _text == RAW_COLS[:-1], f'the text columns of train.csv changed: {_text}!!'


def base_features(d):
    '''One fold's 121 base features for its train, valid and test rows, in the row order of the design matrix.
    fold 하나의 기본 피쳐 121열. 행 순서는 설계행렬과 같다.'''
    out = {}
    for side, frame in (('train', train_fixed), ('valid', train_fixed), ('test', test_fixed)):
        src = frame.set_index('Id').loc[getattr(d, f'ids_{side}')]
        num = pd.DataFrame(getattr(d, f'X_{side}')[:, :N_NUM], columns=d.features[:N_NUM])
        raw = pd.DataFrame({n: src[c].astype(str).to_numpy() for n, c in zip(RAW_NAMES, RAW_COLS)})
        out[side] = pd.concat([num, raw], axis=1)
    return out


def sale_group(frame, ids):
    sc = frame.set_index('Id').loc[ids, 'SaleCondition'].to_numpy()
    assert np.isin(sc, MARKET + NONMARKET).all(), f'unknown sale condition {set(sc) - set(MARKET + NONMARKET)}!!'
    return np.where(np.isin(sc, MARKET), 'market', 'nonmarket')


BASE = {f: base_features(DES[f]) for f in range(5)}
GROUP = {f: dict(train=sale_group(train_fixed, DES[f].ids_train), valid=sale_group(train_fixed, DES[f].ids_valid))
         for f in range(5)}
GROUP_TEST = sale_group(test_fixed, DES[0].ids_test)

print(f'base features {BASE[0]["train"].shape[1]} = {N_NUM} numeric + {len(RAW_COLS)} raw labels')
print(f'\n{"fold":>5}{"context: market":>17}{"non-market":>12}{"valid: market":>15}{"non-market":>12}')
for f in range(5):
    assert BASE[f]['train'].shape[1] == N_NUM + len(RAW_COLS), f'fold {f} base features!!'
    _c, _v = GROUP[f]['train'], GROUP[f]['valid']
    print(f'{f:>5}{int((_c == "market").sum()):>17,}{int((_c == "nonmarket").sum()):>12,}'
          f'{int((_v == "market").sum()):>15,}{int((_v == "nonmarket").sum()):>12,}')
print(f'\ntest: market {int((GROUP_TEST == "market").sum()):,} · non-market {int((GROUP_TEST == "nonmarket").sum()):,}')

base features 121 = 78 numeric + 43 raw labels

 fold  context: market  non-market  valid: market  non-market
    0            1,061         101            255          36
    1            1,054         109            262          28
    2            1,044         119            272          18
    3            1,053         110            263          27
    4            1,054         109            262          28

test: market 1,324 · non-market 135


# 4. Define the model / 모델 정의

### **모델은 판매 조건별로 나뉜 전문가 셋이다**
fold마다 전문가 셋이 같은 기본 피쳐를 받되, 붙는 유전자 열과 컨텍스트의 조합이 서로 다르다.<br>
어느 집에 누구의 예측을 쓸지는 판매 조건이 정한다.

- **TabPFN v2 · 시장** : 기본 피쳐 121열 + 축별 유전자를 받고, 시장 거래 train 행을 컨텍스트로 쓴다.
- **TabICL v2 · 시장** : 기본 피쳐 121열 + ARD · 릿지 유전자를 받고, 시장 거래 train 행을 컨텍스트로 쓴다.
- **TabICL v2 · 비시장** : 입력은 바로 위와 같고, 비시장 거래 train 행을 컨텍스트로 쓴다.

시장 거래 집에는 앞의 두 전문가를 반반 섞은 값을 쓰고, 비시장 거래 집에는 세 번째 전문가의 예측을 그대로 쓴다.<br>
세 전문가 모두 가중치를 학습하지 않는다.<br>
컨텍스트와 입력이 달라지면 예측이 달라질 뿐이다.

## Genes from the saved GP models / 저장된 GP 모델의 유전자

### Primitives and the expression evaluator / 연산자와 수식 평가기

유전자는 DEAP 수식 나무를 문자열로 적은 것이다.<br>
문자열을 다시 나무로 만들고, 나무를 모든 행에 대해 열 단위로 계산한다.<br>
이전 노트북의 진화 루프가 개체를 평가할 때 쓰던 연산자와 평가기를 그대로 쓴다.

- **보호 연산자** : 0으로 나누거나 음수에 로그를 씌워도 오류 없이 유한한 값을 돌려준다. 분모의 절댓값이 0.001 이하이면 나눗셈은 1, |x| ≤ 0.001이면 log는 0을 내고, sqrt는 |x|를 쓴다.
- **터미널** : 설계행렬 89열이다. 나무 속 상수는 숫자 그대로 읽는다.

In [17]:
# Protected operators, the primitive set the genes were written in, and the column-wise expression evaluator.
# 보호 연산자, 유전자를 적은 프리미티브 집합(pset), 그리고 수식을 열 단위로 계산하는 평가기.
#   protected_div/log/sqrt : 0 나누기·음수 로그에서 예외 대신 유한값을 돌려준다
#   make_pset              : 터미널 89개 + 연산자 8개. 문자열로 저장한 유전자를 나무로 되살릴 때 쓴다
#   Evaluator.forward      : 나무 하나를 후위 순회로 한 번에 계산한다

PROTECT_EPS = 0.001


# Protected operators return a finite value instead of raising, so a gene never dies of a zero divide.
# 보호 연산자는 오류를 내는 대신 유한한 값을 돌려준다.
# |분모| <= 0.001이면 나눗셈은 1, |x| <= 0.001이면 log는 0, sqrt는 |x|를 쓴다.
def protected_div(a, b):
    if isinstance(b, np.ndarray):
        out = np.ones_like(b)
        np.divide(a, b, out=out, where=np.abs(b) > PROTECT_EPS)
        return out
    if abs(b) > PROTECT_EPS:
        return a / b
    return np.ones_like(a) if isinstance(a, np.ndarray) else np.float64(1.0)

def protected_log(a):
    if isinstance(a, np.ndarray):
        absa = np.abs(a)
        out = np.zeros_like(absa)
        np.log(absa, out=out, where=absa > PROTECT_EPS)
        return out
    absa = abs(a)
    return np.log(absa) if absa > PROTECT_EPS else np.float64(0.0)

def protected_sqrt(a):
    return np.sqrt(np.abs(a))

PRIMITIVES = {'add': operator.add, 'sub': operator.sub,
              'mul': operator.mul, 'div': protected_div,
              'log': protected_log, 'sqrt': protected_sqrt,
              'max': np.maximum, 'min': np.minimum}
PRIMITIVES = {k: (v, 1 if k in ('log', 'sqrt') else 2)
              for k, v in PRIMITIVES.items()}

K_COL, K_CONST, K_PRIM = 0, 1, 2


# 89 terminals and 8 operators. A constant inside a saved gene is read back as a plain number.
# 터미널 89개와 연산자 8개. 저장된 유전자 속 상수는 숫자 그대로 읽힌다.
def make_pset(names):
    pset = gp.PrimitiveSet('GENES', len(names))
    for name, (fn, arity) in PRIMITIVES.items():
        pset.addPrimitive(fn, arity, name=name)
    pset.renameArguments(**{f'ARG{i}': n for i, n in enumerate(names)})
    return pset


# 노드 이름 → (종류, 값). 터미널 이름은 renameArguments 뒤에도 ARGi로 남으므로 value로 열 번호를 찾는다.
def node_table(pset):
    table = {}
    for terms in pset.terminals.values():
        for term in terms:
            table[term.name] = (K_COL, pset.arguments.index(term.value))
    for prims in pset.primitives.values():
        for prim in prims:
            table[prim.name] = (K_PRIM, prim.name)
    return table


# 나무를 (종류, 값) 배열로 바꾼다. forward가 이것을 읽는다. 표에 없는 노드는 상수다.
def compile_tree(tree, table):
    T = len(tree)
    kind = [0] * T
    payload = [None] * T
    kids = [()] * T
    stack = []
    for i in range(T - 1, -1, -1):
        node = tree[i]
        k, v = table.get(node.name, (K_CONST, None))
        if k == K_CONST:
            kind[i], payload[i] = K_CONST, float(node.value)
        elif k == K_PRIM:
            kind[i], payload[i] = K_PRIM, v
            kids[i] = tuple(stack.pop() for _ in range(node.arity))
        else:
            kind[i], payload[i] = k, v
        stack.append(i)
    return kind, payload, kids

def _apply(name, args):
    if name == 'add':
        return args[0] + args[1]
    if name == 'sub':
        return args[0] - args[1]
    if name == 'mul':
        return args[0] * args[1]
    if name == 'div':
        return protected_div(args[0], args[1])
    if name == 'log':
        return protected_log(args[0])
    if name == 'sqrt':
        return protected_sqrt(args[0])
    if name == 'max':
        return np.maximum(args[0], args[1])
    if name == 'min':
        return np.minimum(args[0], args[1])
    raise KeyError(name)


# Trees are evaluated column-wise over all rows at once, postfix order, one pass.
# 나무를 모든 행에 대해 열 단위로 한 번에 계산한다. 후위 순회 한 번이면 된다.
class Evaluator:

    def __init__(self, X, table):
        self.n = X.shape[0]
        self.cols = [np.ascontiguousarray(X[:, j]) for j in range(X.shape[1])]
        self.table = table
        self._ones = np.ones(self.n)

    def forward(self, tree):
        kind, payload, kids = compile_tree(tree, self.table)
        val = [None] * len(kind)
        with np.errstate(all='ignore'):
            for i in range(len(kind) - 1, -1, -1):
                k = kind[i]
                if k == K_COL:
                    val[i] = self.cols[payload[i]]
                elif k == K_CONST:
                    val[i] = payload[i] * self._ones
                else:
                    val[i] = _apply(payload[i], [val[c] for c in kids[i]])
        out = val[0]
        return out if isinstance(out, np.ndarray) and out.shape == (self.n,) else np.broadcast_to(out, (self.n,))


# The genes of one model become the columns of one matrix (n × genes).
# 모델 하나의 유전자들이 행렬 하나의 열이 된다(n × 유전자 수).
def gene_matrix(ev, trees):
    cols = [ev.forward(t) for t in trees]
    return np.column_stack(cols) if cols else np.zeros((ev.n, 0))

### Gene columns / 유전자 열

GP 모델 하나의 유전자 값은 GP가 계산하던 방식 그대로 만든다.

1. 그 fold의 설계행렬에서 나무를 계산한다(train · valid · test 행 전부).
2. 유전자마다 train 행에서 낸 최솟값과 최댓값(`g_lo`, `g_hi`)으로 자른다. GP 모델이 학습하지 않은 행은 여기서 잘린다. 축별 GP에게는 다른 축의 행이 모두 처음 보는 행이다.
3. 유한하지 않은 값은 그 유전자의 train 평균으로 채운다.
4. 결정자가 끈 유전자를 빼고, 살아 있는 유전자만 남긴다.

열 이름은 `GENE_{런}_{번호}`이고, 번호는 그 GP 모델 안에서의 유전자 순서다.<br>
전문가에게 들어가는 표는 수치 78열, 원본 범주형 43열, 유전자 열 순으로 놓는다.<br>
TabPFN은 열 순서를 섞어서 읽지만 섞는 방식이 원래 열 순서에 묶여 있으므로, 같은 예측을 얻으려면 순서를 지켜야 한다.

In [18]:
# A saved GP model, ready to compute its genes: the trees parsed back, the live genes, each gene's training range,
# and its training mean for the rare non-finite value. The coefficients are read only by the check in section 6.
# 저장된 GP 모델을 유전자 계산에 쓸 수 있게 되살린다. 나무, 살아 있는 유전자, 유전자별 train 범위, 그리고 드물게 나오는
# 비유한 값을 채울 train 평균을 준비한다. 계수(β)는 6절의 점검에서만 쓴다.
def load_gp(rec):
    pset = make_pset(rec['terminals'])
    bound = lambda xs, fill: np.array([fill if x is None else x for x in xs], float)    # noqa: E731
    return dict(rec, trees=[gp.PrimitiveTree.from_string(g, pset) for g in rec['genes']], table=node_table(pset),
                alive=np.asarray(rec['alive'], bool), mu=np.asarray(rec['mu'], float),
                sd=np.asarray(rec['sd'], float), beta=np.asarray(rec['beta'], float),
                g_lo=bound(rec['g_lo'], -np.inf), g_hi=bound(rec['g_hi'], np.inf))


# One model's gene values on one block of rows, exactly as the GP computed them: clipped to each gene's training range
# (rows the GP never trained on are clipped here), non-finite values set to the training mean, switched-off genes dropped.
# 유전자 값은 GP가 계산하던 방식 그대로 만든다. 유전자마다 train 범위로 자르고(GP가 학습하지 않은 행은 여기서 잘린다),
# 비유한 값은 train 평균으로 채우고, 결정자가 끈 유전자는 뺀다. 계수는 곱하지 않는다.
def gene_values(m, X):
    G = gene_matrix(Evaluator(X, m['table']), m['trees'])
    G = np.clip(G, m['g_lo'], m['g_hi'])
    G = np.where(np.isfinite(G), G, m['mu'][None, :])
    return np.ascontiguousarray(G[:, m['alive']])


# An expert's table: the 121 base features, then the gene columns of the fold's GP models in the given order.
# 전문가에게 들어가는 표. 기본 피쳐 121열 뒤에 그 fold GP 모델들의 유전자 열을 주어진 순서대로 붙인다.
def material(f, runs):
    d, out = DES[f], {}
    for side in ('train', 'valid', 'test'):
        blocks = [BASE[f][side]]
        for r in runs:
            m = GP_MODELS[r, f]
            blocks.append(pd.DataFrame(gene_values(m, getattr(d, f'X_{side}')),
                                       columns=[f'GENE_{r}_{j}' for j in np.flatnonzero(m['alive'])]))
        out[side] = pd.concat(blocks, axis=1)
    return out


GP_MODELS = {k: load_gp(rec) for k, rec in GP_REC.items()}
for (_r, _f), _m in GP_MODELS.items():
    assert _m['terminals'] == DES[_f].features, f'GP {_r} fold {_f} was written over other columns than the design!!'

MATERIAL = {f: {gs: material(f, runs) for gs, runs in GENE_SETS.items()} for f in range(5)}

print(f'{"fold":>5}' + ''.join(f'{gs + " genes":>18}{"columns":>9}' for gs in GENE_SETS))
for f in range(5):
    _cells = []
    for gs in GENE_SETS:
        _X = MATERIAL[f][gs]
        _g = [c for c in _X['train'].columns if c.startswith('GENE_')]
        assert all(np.isfinite(_X[s][_g].to_numpy()).all() for s in ('train', 'valid', 'test')), \
            f'fold {f} {gs}: non-finite gene values!!'
        _cells.append(f'{len(_g):>18}{_X["train"].shape[1]:>9}')
    print(f'{f:>5}' + ''.join(_cells))

 fold        axis genes  columns   ard_ridge genes  columns
    0                42      163                43      164
    1                42      163                51      172
    2                41      162                49      170
    3                52      173                48      169
    4                51      172                47      168


## The experts: TabPFN v2 and TabICL v2 / 전문가: TabPFN v2와 TabICL v2

두 모델 모두 저자 기본 설정으로 부르고, 씨앗만 `random_state=0`으로 고정한다.

- **컨텍스트** : 그 fold의 train 행 가운데 한 판매 조건 묶음의 행과 그 log 가격이다. `fit`은 컨텍스트를 담고 전처리를 맞출 뿐, 가중치는 바꾸지 않는다.
- **멤버** : 둘 다 표를 조금씩 바꿔 읽은 예측 8개를 평균한다. TabPFN은 `n_estimators='auto'`로 두며, v2에서는 8개다.
- **범주형 열** : TabPFN에는 원본 범주형 43열의 위치를 `categorical_features_indices`로 알려 준다. TabICL은 글자 열을 알아서 범주로 읽는다.
- **예측값** : 예측 분포의 평균을 log 가격의 예측값으로 쓴다.
- **예측하는 행** : 전문가마다 그 fold의 valid 행과 test 행을 전부 한 번씩 예측하고, 섞을 때는 자기 묶음의 행만 가져다 쓴다. 예측하는 행끼리는 서로 참조하지 않으므로, 다른 묶음의 행이 함께 있어도 예측은 달라지지 않는다.

`tabpfn` 9.2.0의 기본 모델은 v2가 아니다.<br>
그래서 `ModelVersion.V2`로 v2를 지정하고, 1절에서 받아 확인한 가중치 파일의 경로를 직접 넘긴다.

In [19]:
# TabPFN v2 is asked for by name, since tabpfn 9.2.0 defaults to a newer model, and gets the weight file checked in section 1.
# tabpfn 9.2.0은 기본 모델이 v2가 아니라서 TabPFN v2를 이름으로 지정하고, 1절에서 해시를 확인한 가중치 파일을 직접 넘긴다.
def fit_tabpfn(X_ctx, y_ctx):
    cats = [list(X_ctx.columns).index(c) for c in RAW_NAMES]      # 원본 범주형 43열의 위치
    m = TabPFNRegressor.create_default_for_version(
        ModelVersion.V2, model_path=str(WEIGHT_PATH['tabpfn']), n_estimators='auto',
        categorical_features_indices=cats, device='auto', random_state=MODEL_SEED)
    return m.fit(X_ctx, y_ctx)


# TabICL v2 reads text columns as categories by itself. Nothing is downloaded here; the file is the one checked in section 1.
# TabICL v2는 글자 열을 알아서 범주로 읽는다. 여기서는 내려받지 않고 1절에서 확인한 파일을 쓴다.
def fit_tabicl(X_ctx, y_ctx):
    m = TabICLRegressor(model_path=str(WEIGHT_PATH['tabicl']), allow_auto_download=False,
                        n_estimators=N_MEMBERS, random_state=MODEL_SEED)
    return m.fit(X_ctx, y_ctx)


FIT = {'tabpfn': fit_tabpfn, 'tabicl': fit_tabicl}


# One expert on one fold. The context is the fold's train rows of one sale group, and fitting only stores it.
# The expert predicts every valid and test row of the fold (rows being predicted never see each other), each prediction is
# clipped to the fold's training log-price range, and the mixing rule later takes the rows of the expert's own group.
# 전문가 하나를 fold 하나에서 돌린다. 컨텍스트는 그 fold의 train 행 가운데 한 판매 조건 묶음이고, fit은 이 컨텍스트를 담기만 한다.
# 그 fold의 valid · test 행을 전부 예측하고(예측하는 행끼리는 서로 참조하지 않는다), 예측값은 그 fold의 train log 가격 범위로 자른다.
# 섞을 때는 그중 자기 묶음의 행만 쓴다.
def run_expert(f, spec):
    d, X = DES[f], MATERIAL[f][spec['genes']]
    ctx = GROUP[f]['train'] == spec['context']
    m = FIT[spec['model']](X['train'][ctx].reset_index(drop=True), d.y_train[ctx])
    lo, hi = float(d.y_train.min()), float(d.y_train.max())
    return dict(valid=np.clip(np.asarray(m.predict(X['valid']), float), lo, hi),
                test=np.clip(np.asarray(m.predict(X['test']), float), lo, hi),
                n_context=int(ctx.sum()), n_cols=int(X['train'].shape[1]),
                members=int(getattr(m, 'n_estimators_', N_MEMBERS)))

## The mixing rule / 섞는 규칙

fold 하나의 예측값은 판매 조건에 따라 정한다.

$$\hat y_i = \begin{cases} 0.5\,\hat y^{\,\text{TabPFN}}_i + 0.5\,\hat y^{\,\text{TabICL, market}}_i & \text{market sale} \\[2pt] \hat y^{\,\text{TabICL, non-market}}_i & \text{non-market sale} \end{cases}$$

각 전문가의 예측값은 섞기 전에 그 fold의 train log 가격 범위로 자른다.<br>
제출 파일은 다섯 fold의 예측을 log 공간에서 평균한 값이다.

In [20]:
# The mixing rule: market rows take W_MARKET · TabPFN + (1 − W_MARKET) · TabICL, non-market rows take TabICL alone.
# 섞는 규칙: 시장 거래 행은 W_MARKET · TabPFN + (1 − W_MARKET) · TabICL, 비시장 거래 행은 TabICL 단독이다.
def mix(parts, group):
    market = W_MARKET * parts['tabpfn_market'] + (1 - W_MARKET) * parts['tabicl_market']
    return np.where(group == 'market', market, parts['tabicl_nonmarket'])

# 5. Predict / 예측

fold마다 전문가 셋을 차례로 돌린다.<br>
fold 다섯 개에 전문가 셋이니 예측은 모두 15번이다.<br>

fold마다 섞은 예측의 valid 점수를 축별로 출력한다.<br>
축별 점수가 fold마다 크게 다른 것은 정상이다.<br>

In [21]:
# Three experts per fold over five folds, 15 predictions in all. Nothing is trained: fit only stores the context,
# and each prediction is a single pass through the model.
# fold마다 전문가 셋, fold 다섯 개라 예측은 모두 15번이다. 학습은 없다. fit은 컨텍스트를 담기만 하고, 예측은 모델을 한 번 통과하면 끝난다.
rmse = lambda p, y: float(np.sqrt(np.mean((np.asarray(p) - y) ** 2)))    # noqa: E731

_t0 = time.time()
PARTS, ENS = {}, {}
for f in range(5):
    PARTS[f] = {}
    for _name, _spec in EXPERTS.items():
        _t = time.time()
        PARTS[f][_name] = _p = run_expert(f, _spec)
        print(f'  fold {f} · {_name:<16}: context {_p["n_context"]:>5,} rows · {_p["n_cols"]} columns '
              f'· {_p["members"]} members · {time.time() - _t:4.1f} s', flush=True)
    # 섞는다. valid는 이 fold의 valid 행, test는 1,459행 전부다
    ENS[f] = dict(valid=mix({k: v['valid'] for k, v in PARTS[f].items()}, GROUP[f]['valid']),
                  test=mix({k: v['test'] for k, v in PARTS[f].items()}, GROUP_TEST))
    _y, _g = DES[f].y_valid, GROUP[f]['valid']
    print(f'  fold {f} · mix valid: market {rmse(ENS[f]["valid"][_g == "market"], _y[_g == "market"]):.4f} '
          f'· non-market {rmse(ENS[f]["valid"][_g == "nonmarket"], _y[_g == "nonmarket"]):.4f} '
          f'· all {rmse(ENS[f]["valid"], _y):.4f}\n', flush=True)

print(f'*** {(time.time() - _t0) / 60:.1f} min · {sum(len(p) for p in PARTS.values())} expert runs · five folds done ***')

  fold 0 · tabpfn_market   : context 1,061 rows · 163 columns · 8 members · 17.8 s
  fold 0 · tabicl_market   : context 1,061 rows · 164 columns · 8 members ·  5.5 s
  fold 0 · tabicl_nonmarket: context   101 rows · 164 columns · 8 members ·  2.8 s
  fold 0 · mix valid: market 0.1099 · non-market 0.2193 · all 0.1286

  fold 1 · tabpfn_market   : context 1,054 rows · 163 columns · 8 members · 16.0 s
  fold 1 · tabicl_market   : context 1,054 rows · 172 columns · 8 members ·  5.5 s
  fold 1 · tabicl_nonmarket: context   109 rows · 172 columns · 8 members ·  3.0 s
  fold 1 · mix valid: market 0.0951 · non-market 0.2372 · all 0.1167

  fold 2 · tabpfn_market   : context 1,044 rows · 162 columns · 8 members · 16.2 s
  fold 2 · tabicl_market   : context 1,044 rows · 170 columns · 8 members ·  5.5 s
  fold 2 · tabicl_nonmarket: context   119 rows · 170 columns · 8 members ·  3.0 s
  fold 2 · mix valid: market 0.0938 · non-market 0.1683 · all 0.1001

  fold 3 · tabpfn_market   : context 1,053 

# 6. Validation / 검증

세 가지를 확인한다.

1. **유전자 열** : GP 모델 20개가 이 노트북이 계산한 유전자 값으로 기록된 valid 점수를 재현하는지 본다. 유전자 값에 그 GP의 계수를 곱해 더한 예측이 기록과 같아야 한다. 번들, 전처리, 수식 평가기 가운데 하나라도 어긋나면 여기서 드러난다.
2. **섞은 예측** : 축마다 각 전문가의 예측과 섞은 예측의 valid 점수를 본다. 다섯 fold의 valid 행을 합쳐서 잰다.
3. **제출 형식** : `sample_submission`과 같은 Id 순서인지, 예측값이 모두 유한한지, 가격 범위가 정상인지 본다.

valid 점수가 기록에서 크게 벗어나면 제출을 막는다.

In [22]:
# Check 2 of 4: do the gene values this notebook computed reproduce the saved GP models?
# 검증 2/4: 이 노트북이 계산한 유전자 값이 저장된 GP 모델을 재현하는가?
# Standardised and multiplied by the GP's own coefficients, the gene values must give the validation score recorded with
# the model, on the validation rows of the sales it was evolved on. A wrong bundle, preprocessing or evaluator shows up here.
# 유전자 값을 표준화해 그 GP의 계수를 곱해 더하면, 그 GP가 진화한 판매 조건의 valid 행에서 모델과 함께 기록된 valid 점수가
# 나와야 한다. 번들, 전처리, 수식 평가기 가운데 하나라도 어긋나면 여기서 드러난다.
def gp_predict(m, G):
    a = m['alive']
    return np.clip(m['intercept'] + ((G - m['mu'][a]) / m['sd'][a]) @ m['beta'][a], m['y_lo'], m['y_hi'])


print(f'{"GP model":<11}' + ''.join(f'{f"fold {f}":>18}' for f in range(5)) + '   (this notebook / recorded)')
_worst = 0.0
for _r in GP_RUNS:
    _cells = []
    for f in range(5):
        _m, d = GP_MODELS[_r, f], DES[f]
        _rows = np.ones(len(d.y_valid), bool) if _m['rows'] == 'all' else GROUP[f]['valid'] == _m['rows']
        _got = rmse(gp_predict(_m, gene_values(_m, d.X_valid))[_rows], d.y_valid[_rows])
        _worst = max(_worst, abs(_got - _m['valid_rmse']))
        assert abs(_got - _m['valid_rmse']) < 1e-6, \
            f'GP {_r} fold {f}: valid {_got:.6f} from the genes, {_m["valid_rmse"]:.6f} recorded!!'
        _cells.append(f'{_got:.4f} / {_m["valid_rmse"]:.4f}')
    print(f'{_r:<11}' + ''.join(f'{c:>18}' for c in _cells))
print(f'largest gap {_worst:.1e}: every GP model reproduces its recorded validation score from these genes\n')

# The mix, axis by axis, on the 1,451 validation rows pooled over the folds. Each row is predicted by the fold that held it out.
# 섞은 예측을 축마다 본다. 다섯 fold의 valid 행 1,451개를 합쳐서 재며, 각 행은 그 행을 컨텍스트에 넣지 않은 fold가 예측한 값이다.
Y_VALID = np.concatenate([DES[f].y_valid for f in range(5)])
G_VALID = np.concatenate([GROUP[f]['valid'] for f in range(5)])
P_VALID = {k: np.concatenate([PARTS[f][k]['valid'] for f in range(5)]) for k in EXPERTS}
E_VALID = np.concatenate([ENS[f]['valid'] for f in range(5)])
VALID_POOLED = rmse(E_VALID, Y_VALID)

print(f'{"valid rows":<22}' + ''.join(f'{k:>18}' for k in EXPERTS) + f'{"mix":>9}')
for _grp, _label in (('market', 'market'), ('nonmarket', 'non-market')):
    _sel = G_VALID == _grp
    _cells = [f'{rmse(P_VALID[k][_sel], Y_VALID[_sel]):>18.4f}' if s['context'] == _grp else f'{"-":>18}'
              for k, s in EXPERTS.items()]
    print(f'{_label + f" ({int(_sel.sum()):,})":<22}' + ''.join(_cells) + f'{rmse(E_VALID[_sel], Y_VALID[_sel]):>9.4f}')
print(f'{f"all ({len(Y_VALID):,})":<22}' + ''.join(f'{"":>18}' for _ in EXPERTS) + f'{VALID_POOLED:>9.4f}\n')

# The submission side: the test Id order, finiteness and the training price range.
# 제출 쪽 확인: test Id 순서, 유한성, train 가격 범위.
Y_LO = float(min(DES[f].y_train.min() for f in range(5)))
Y_HI = float(max(DES[f].y_train.max() for f in range(5)))
print(f'training log-price range [{Y_LO:.3f}, {Y_HI:.3f}] = ${np.exp(Y_LO):,.0f} ~ ${np.exp(Y_HI):,.0f}')

sub_head = pd.read_csv(COMP_DIR / 'sample_submission.csv', nrows=0)
assert list(sub_head.columns) == ['Id', 'SalePrice'], 'sample_submission columns changed!!'
sample_ids = pd.read_csv(COMP_DIR / 'sample_submission.csv').Id.to_numpy()

for f in range(5):
    assert DES[f].ids_test.tolist() == sample_ids.tolist(), f'fold {f} test Id order != sample_submission!!'
    for _k, _p in PARTS[f].items():
        assert np.isfinite(_p['valid']).all() and np.isfinite(_p['test']).all(), f'fold {f} {_k}: non-finite predictions!!'
    _t = ENS[f]['test']
    assert Y_LO <= float(np.median(_t)) <= Y_HI, f'fold {f} median prediction {np.median(_t):.2f} is off the price range!!'
    print(f'  fold {f}: test predictions [{_t.min():.2f}, {_t.max():.2f}] · median {np.median(_t):.2f}')

print('\n*** Validation passed: the genes reproduce the GP models, the mix holds, Id order matches ***')

GP model               fold 0            fold 1            fold 2            fold 3            fold 4   (this notebook / recorded)
ard           0.1367 / 0.1367   0.1362 / 0.1362   0.1142 / 0.1142   0.1201 / 0.1201   0.1071 / 0.1071
ridge         0.1378 / 0.1378   0.1206 / 0.1206   0.1203 / 0.1203   0.1332 / 0.1332   0.1071 / 0.1071
market        0.1236 / 0.1236   0.1066 / 0.1066   0.1016 / 0.1016   0.1304 / 0.1304   0.0979 / 0.0979
nonmarket     0.2730 / 0.2730   0.3437 / 0.3437   0.2652 / 0.2652   0.2699 / 0.2699   0.2983 / 0.2983
largest gap 2.9e-15: every GP model reproduces its recorded validation score from these genes

valid rows                 tabpfn_market     tabicl_market  tabicl_nonmarket      mix
market (1,314)                    0.0961            0.0958                 -   0.0938
non-market (137)                       -                 -            0.2066   0.2066
all (1,451)                                                                    0.1095

training log-price ra

# 7. Records for the figures / 그림에 쓸 기록

전문가마다의 예측, 섞은 예측, 전문가에게 들어간 열을 파일로 남긴다.<br>
다시 돌리지 않고도 앙상블을 살펴볼 수 있게 하기 위해서다.

| 파일 | 내용 | 쓰임 |
| --- | --- | --- |
| `valid_parts.csv` | valid 행 1,451개의 fold, 판매 조건, 정답(log), 전문가 셋의 예측, 섞은 예측 | 축별 오차, 두 전문가의 오차 비교 |
| `test_parts.csv` | test 행 1,459개의 fold별 전문가 셋 예측과 섞은 예측, 그리고 다섯 fold 평균(제출값의 log) | fold 사이 흩어짐 |
| `run_info.json` | 레시피, fold마다 전문가에게 들어간 열과 컨텍스트 행 수, 유전자 열마다의 식, 가중치 해시, 실행 환경 | 입력 설명 |

확인이 끝나면 세 파일을 `viz_records.zip` 하나로 묶는다.<br>
Kaggle 제출 화면은 출력 폴더의 파일을 알파벳 순으로 보여 주므로, 이름이 `s` 뒤에 오는 zip 하나로 두면 `submission.csv`가 맨 위에 보인다.<br>
가중치 파일은 출력에 들어가지 않는다.

In [23]:
# The records let the run be read without running it again: every expert's predictions, the mix, and what each expert got.
# 다시 돌리지 않고도 런을 살펴볼 수 있게 남긴다. 전문가마다의 예측, 섞은 예측, 전문가에게 들어간 열이다.
VIZ = OUTPUT_DIR / 'viz'          # Kaggle output, downloadable / 캐글 출력, 내려받을 수 있다
VIZ.mkdir(parents=True, exist_ok=True)

# valid: one row per validation house, predicted by the fold that held it out
pd.concat([pd.DataFrame(dict(Id=DES[f].ids_valid, fold=f, sale_group=GROUP[f]['valid'], y_log=DES[f].y_valid,
                             **{k: PARTS[f][k]['valid'] for k in EXPERTS}, mix=ENS[f]['valid']))
           for f in range(5)], ignore_index=True).to_csv(VIZ / 'valid_parts.csv', index=False)

# test: one row per test house, with every expert's prediction on every fold, each fold's mix, and their mean (the submission, in log)
_test = pd.DataFrame(dict(Id=DES[0].ids_test, sale_group=GROUP_TEST))
for f in range(5):
    for _k in EXPERTS:
        _test[f'{_k}_f{f}'] = PARTS[f][_k]['test']
    _test[f'mix_f{f}'] = ENS[f]['test']
_test['mean'] = np.mean([ENS[f]['test'] for f in range(5)], axis=0)
_test.to_csv(VIZ / 'test_parts.csv', index=False)

# what each expert got on each fold, the expression behind every gene column, and the run's environment
(VIZ / 'run_info.json').write_text(json.dumps(dict(
    recipe=dict(experts=EXPERTS, gene_sets=GENE_SETS, w_market=W_MARKET, members=N_MEMBERS, random_state=MODEL_SEED),
    folds={f: {k: dict(context=s['context'], n_context=PARTS[f][k]['n_context'],
                       columns=list(MATERIAL[f][s['genes']]['train'].columns)) for k, s in EXPERTS.items()}
           for f in range(5)},
    genes={f: {f'GENE_{r}_{j}': GP_MODELS[r, f]['genes'][j] for r in GP_RUNS for j in np.flatnonzero(GP_MODELS[r, f]['alive'])}
           for f in range(5)},
    weights={k: dict(w, bytes=WEIGHT_PATH[k].stat().st_size) for k, w in WEIGHTS.items()},
    valid=dict(pooled=VALID_POOLED, market=rmse(E_VALID[G_VALID == 'market'], Y_VALID[G_VALID == 'market']),
               nonmarket=rmse(E_VALID[G_VALID == 'nonmarket'], Y_VALID[G_VALID == 'nonmarket'])),
    env=dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__,
             tabpfn=installed('tabpfn'), tabicl=installed('tabicl'))), indent=1, default=int), encoding='utf-8')

_files = sorted(VIZ.iterdir())
print(f'{"file":<16}{"size":>10}')
for _p in _files:
    assert _p.stat().st_size > 0, f'{_p.name} is empty!!'
    print(f'{_p.name:<16}{_p.stat().st_size / 1024:>7,.0f} KB')

# Kaggle's "Submit to Competition -> Notebook" list is the output tree in alphabetical order. One zip whose name sorts
# after 's' keeps submission.csv on top.
# 캐글 제출 목록은 출력 폴더의 파일을 알파벳 순으로 보여 준다. 이름이 's' 뒤에 오는 zip 하나로 묶어 submission.csv가 맨 위에 보이게 한다.
_zip = ROOT_DIR / 'viz_records.zip'
with zipfile.ZipFile(_zip, 'w', zipfile.ZIP_DEFLATED) as _z:
    for _p in _files:
        _z.write(_p, _p.name)      # 디렉터리 없이 파일 이름만 넣는다 / file names only, no paths
shutil.rmtree(OUTPUT_DIR)

print(f'\n*** Packed -> {_zip.name}  {_zip.stat().st_size / 1e6:.2f} MB ***')
print(f'    output root: submission.csv and {_zip.name} only, no weight file')
print(f'    출력 루트에는 submission.csv와 {_zip.name}뿐이고, 가중치 파일은 출력에 들어가지 않는다')

file                  size
run_info.json       115 KB
test_parts.csv      580 KB
valid_parts.csv     153 KB

*** Packed -> viz_records.zip  0.32 MB ***
    output root: submission.csv and viz_records.zip only, no weight file
    출력 루트에는 submission.csv와 viz_records.zip뿐이고, 가중치 파일은 출력에 들어가지 않는다


# 8. Submission / 제출

### Score gate / 점수 문턱

예측과 기록 저장은 이미 끝났다.<br>
여기서 막는 것은 제출뿐이다.

valid 점수로 리더보드 점수를 예측할 수는 없다.<br>
그래도 기록과 나란히 놓으면 런이 망가졌는지는 알 수 있다.<br>
실행 환경이 다르면 수치가 조금 움직이겠지만, 크게 움직여서는 안 된다.

- **기록 문턱** : 다섯 fold의 valid 행 1,451개를 합친 valid RMSE가 기록 0.1095보다 0.002 이상 나쁘면 제출을 막는다. 기록은 같은 레시피를 같은 fold 표와 같은 가중치로 돌린 런의 값이다.

같은 레시피를 다른 GPU나 float32로 돌리면 합친 valid가 0.0001 안팎 움직인다.<br>
0.002는 그 스무 배이므로, 이 문턱에 걸렸다면 장치 차이가 아니라 어딘가가 망가진 것이다.

In [24]:
# Check 3 of 4: the score gate. The predictions and the records are already done; only the submission is gated.
# 검증 3/4: 점수 문턱. 예측과 기록은 이미 끝났고, 여기서 막을 수 있는 것은 제출뿐이다.
# REF_VALID is the one number that comes from outside this notebook: the same recipe's pooled valid on the same fold table.
# 이 노트북 밖에서 가져온 값은 REF_VALID 하나뿐이다. 같은 레시피를 같은 fold 표로 돌린 런의 합친 valid다.
REF_VALID = 0.1095      # the record: pooled valid RMSE over the 1,451 validation rows
                        # 기록. valid 행 1,451개를 합친 valid RMSE
REF_TOL = 0.002         # another GPU or float32 moves it by about 0.0001, so twenty times that means breakage, not hardware
                        # 다른 GPU나 float32로 돌리면 0.0001 안팎 움직인다. 그 스무 배를 넘으면 장치 차이가 아니라 어딘가 망가진 것이다

gap = VALID_POOLED - REF_VALID
print(f'pooled valid {VALID_POOLED:.4f} · record {REF_VALID:.4f} · gap {gap:+.4f} (tolerance {REF_TOL})')

assert gap < REF_TOL, \
    f'pooled valid {VALID_POOLED:.4f} is {gap:+.4f} off the record, something broke, submission blocked!!'
GATE_OK = True
print('\n*** Score gate passed: within tolerance of the record ***')

pooled valid 0.1095 · record 0.1095 · gap +0.0000 (tolerance 0.002)

*** Score gate passed: within tolerance of the record ***


### Write submission.csv / 제출 파일 저장

In [25]:
# Check 4 of 4: the submission contract. The ensemble is the mean of the five fold predictions in log space.
# 검증 4/4: 제출 형식. 앙상블은 fold 다섯 개의 예측을 log 공간에서 평균한 값이다.
assert GATE_OK, 'the score gate has not passed, submission blocked!!'

P = np.array([ENS[f]['test'] for f in range(5)])
ens = P.mean(axis=0)

assert P.shape == (5, len(sample_ids)), f'prediction block {P.shape}!!'
assert np.isfinite(ens).all(), 'non-finite values in the ensemble!!'
# Each fold is already clipped to its own training range, so the mean cannot leave the overall one.
# fold마다 자기 train 범위로 이미 잘랐으니 평균은 전체 범위를 벗어날 수 없다.
assert (ens >= Y_LO).all() and (ens <= Y_HI).all(), \
    f'ensemble leaves the training price range [{ens.min():.2f}, {ens.max():.2f}]!!'

submission = pd.DataFrame({'Id': sample_ids, 'SalePrice': np.exp(ens)})

assert list(submission.columns) == ['Id', 'SalePrice'], 'submission columns != contract!!'
assert len(submission) == len(sample_ids), 'row count mismatch!!'
assert submission.notna().all().all(), 'NaN detected in submission!!'
assert submission.SalePrice.gt(0).all(), 'non-positive price!!'

submission.to_csv(SUB_PATH, index=False)
print(f'fold valid    : ' + ', '.join(f'{rmse(ENS[f]["valid"], DES[f].y_valid):.4f}' for f in range(5)))
print(f'spread between fold predictions (mean sd): {P.std(axis=0).mean():.4f}')
print(f'*** Submission written -> {SUB_PATH}  {len(submission):,} rows ***')
print(f'    SalePrice [{submission.SalePrice.min():,.0f}, {submission.SalePrice.max():,.0f}] '
      f'median {submission.SalePrice.median():,.0f}')

fold valid    : 0.1286, 0.1167, 0.1001, 0.1069, 0.0915
spread between fold predictions (mean sd): 0.0211
*** Submission written -> /kaggle/working/submission.csv  1,459 rows ***
    SalePrice [36,905, 559,732] median 157,606


In [26]:
submission.head()

,Id,SalePrice
0,1461,123626.496114
1,1462,161195.765367
2,1463,187482.897323
3,1464,195763.340137
4,1465,185548.426060


In [27]:
print('*** Done. ***')

*** Done. ***


# 끝.